In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### ***This project utilizes the UCF-Crime Dataset(https://www.kaggle.com/datasets/odins0n/ucf-crime-dataset) available on Kaggle. From this dataset, four categories of videos-Burglary, Stealing, Shoplifting, and Robbery-were selected and combined to form a theft-focused dataset for analysis and model training.***

# ***Stage 0 – Video Preprocessing (Code)***

In [ ]:
# ===============================
# Stage 0 : Video Preprocessing
# ===============================

import cv2
import os
import glob

# Input base directory (update if different)
base_dir = "/kaggle/input/crime-data/theft"
output_dir = "/kaggle/working/frames"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Parameters
frame_interval = 10   # extract every 10th frame
frame_size = (224, 224)  # resize frames

# Get all video paths (all categories)
video_paths = glob.glob(f"{base_dir}/*/*.mp4")
print(f"Total videos found: {len(video_paths)}")

# Process each video
for vid_path in video_paths:
    cap = cv2.VideoCapture(vid_path)
    vid_name = os.path.splitext(os.path.basename(vid_path))[0]
    category = vid_path.split("/")[-2]

    save_dir = os.path.join(output_dir, category, vid_name)
    os.makedirs(save_dir, exist_ok=True)

    frame_id, saved = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % frame_interval == 0:
            frame = cv2.resize(frame, frame_size)
            frame_file = os.path.join(save_dir, f"frame_{saved:04d}.jpg")
            cv2.imwrite(frame_file, frame)
            saved += 1
        frame_id += 1

    cap.release()
    print(f"[✔] {category}/{vid_name} → {saved} frames")

print("✅ All videos processed successfully.")

In [ ]:
import shutil

# Path where your processed frames were saved
frames_dir = "/kaggle/working/frames"
zip_path = "/kaggle/working/theft_frames.zip"

# Create zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', frames_dir)
print("✅ Zipped successfully!")

# Download (for Kaggle)
from IPython.display import FileLink
FileLink(zip_path)

In [ ]:
import os
import shutil

src_zip = "/kaggle/working/theft_frames.zip"
dst_dir = "/kaggle/outputs"
dst_zip = os.path.join(dst_dir, "theft_frames.zip")

# ✅ create outputs folder if it doesn’t exist
os.makedirs(dst_dir, exist_ok=True)

# ✅ copy zip file to outputs
if os.path.exists(src_zip):
    shutil.copy(src_zip, dst_zip)
    print("✅ Moved successfully! Now download it from 'Output Files' tab on the right.")
else:
    print("❌ Zip file not found! Make sure you ran the zipping cell first.")

In [ ]:
import shutil
from IPython.display import FileLink
import IPython

zip_path = "/kaggle/working/theft_frames.zip"

# If zip doesn't exist, create it
if not os.path.exists(zip_path):
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', "/kaggle/working/frames")

# ✅ Force browser download automatically
IPython.display.display(FileLink(zip_path))
print("👇 If it doesn't start automatically, right-click link > Save As")

# ***Stage 1 — Feature Extraction Notebook Model 1***

In [ ]:
# =========================================================
# Stage 1: Feature Extraction using VideoMAE (on theft-frames)
# =========================================================

import torch
from torchvision import transforms
from transformers import VideoMAEImageProcessor, VideoMAEModel
from PIL import Image
import os
from tqdm import tqdm

# ---------------------------
# Paths
# ---------------------------
BASE_PATH = "/kaggle/input/theft-frames"
SAVE_PATH = "/kaggle/working/features"
os.makedirs(SAVE_PATH, exist_ok=True)

# ---------------------------
# Load Pretrained VideoMAE
# ---------------------------
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
model = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").eval().to("cuda")

# ---------------------------
# Preprocessing Function
# ---------------------------
def load_frames(folder, every_n=10, max_frames=16):
    frames = []
    all_imgs = sorted(os.listdir(folder))
    for i, f in enumerate(all_imgs):
        if i % every_n == 0:
            img = Image.open(os.path.join(folder, f)).convert("RGB")
            frames.append(img)
        if len(frames) >= max_frames:
            break
    return frames

# ---------------------------
# Process All Categories
# ---------------------------
categories = ["Burglary", "Robbery", "Shoplifting", "Stealing"]

for category in categories:
    cat_path = os.path.join(BASE_PATH, category)
    print(f"🔹 Processing category: {category}")

    cat_out = os.path.join(SAVE_PATH, category)
    os.makedirs(cat_out, exist_ok=True)

    for vid in tqdm(os.listdir(cat_path)):
        vid_folder = os.path.join(cat_path, vid)
        if not os.path.isdir(vid_folder):
            continue

        frames = load_frames(vid_folder, every_n=1)
        if not frames:
            continue

        inputs = processor(frames, return_tensors="pt").to("cuda")
        with torch.no_grad():
            outputs = model(**inputs)
            features = outputs.last_hidden_state.mean(dim=1).cpu()

        torch.save(features, os.path.join(cat_out, f"{vid}.pt"))

print("✅ All features extracted and saved to /kaggle/working/features")

# ---------------------------
# (Optional) Zip features for download or later dataset upload
# ---------------------------
import shutil
shutil.make_archive("/kaggle/working/theft_features", 'zip', SAVE_PATH)
print("📦 Saved as theft_features.zip — check 'Output Files' to download or reuse in next stage")

# ***Stage 1 – Model 2: ResNet Baseline***

In [ ]:
!pip install decord -q

In [ ]:
# =========================================================
# Stage 1 (Alternative Non-Transformer Model): ResNet50 + Temporal Pooling
# =========================================================

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os
from tqdm import tqdm

# ---------------------------
# Paths
# ---------------------------
BASE_PATH = "/kaggle/input/theft-frames"
SAVE_PATH = "/kaggle/working/features_resnet"
os.makedirs(SAVE_PATH, exist_ok=True)

# ---------------------------
# Load pretrained ResNet50
# ---------------------------
resnet = models.resnet50(weights="IMAGENET1K_V1")
resnet = nn.Sequential(*list(resnet.children())[:-1])  # remove classification head
resnet.eval().to("cuda")

# ---------------------------
# Transformations
# ---------------------------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ---------------------------
# Process All Categories
# ---------------------------
categories = ["Burglary", "Robbery", "Shoplifting", "Stealing"]

for category in categories:
    print(f"🔹 Processing category: {category}")
    cat_path = os.path.join(BASE_PATH, category)
    cat_out = os.path.join(SAVE_PATH, category)
    os.makedirs(cat_out, exist_ok=True)

    for vid in tqdm(os.listdir(cat_path)):
        vid_folder = os.path.join(cat_path, vid)
        if not os.path.isdir(vid_folder):
            continue

        frames = sorted([f for f in os.listdir(vid_folder) if f.lower().endswith(('.jpg', '.png'))])
        clip_feats = []

        for f in frames[::5]:  # every 5th frame for speed
            img = Image.open(os.path.join(vid_folder, f)).convert("RGB")
            inp = transform(img).unsqueeze(0).to("cuda")
            with torch.no_grad():
                feat = resnet(inp).squeeze().cpu()
            clip_feats.append(feat)

        if not clip_feats:
            continue

        # Temporal pooling across frames
        clip_feats = torch.stack(clip_feats).mean(dim=0)
        torch.save(clip_feats, os.path.join(cat_out, f"{vid}.pt"))

print("✅ All ResNet features extracted and saved to /kaggle/working/features_resnet")

# ---------------------------
# Zip for upload or next stage
# ---------------------------
import shutil
shutil.make_archive("/kaggle/working/theft_features_resnet", 'zip', SAVE_PATH)
print("📦 Saved as theft_features_resnet.zip — check 'Output Files'")

# ***Stage 1 Evaluation***

In [ ]:
import os

VIT_PATH = "/kaggle/input/theft-features"
CNN_PATH = "/kaggle/input/theft-features-resnet"

for path, name in [(VIT_PATH, "VideoMAE"), (CNN_PATH, "ResNet")]:
    print(f"\n📁 Checking {name} folder:")
    for root, dirs, files in os.walk(path):
        print(f"  {root} -> {len(files)} files")
        if len(files) > 0:
            print("   Example files:", files[:5])
        break  # only top level

In [ ]:
# =========================================================
# 🔹 Stage 1 Evaluation: Feature Quality & Summary
# =========================================================

import os
import torch
import numpy as np
from tqdm import tqdm

# -------------------------------
# Feature directories
# -------------------------------
VIT_PATH = "/kaggle/input/theft-features"
CNN_PATH = "/kaggle/input/theft-features-resnet"

# -------------------------------
# Helper function to load all features from subfolders
# -------------------------------
def load_all_features(base_path):
    feature_dict = {}
    for category in sorted(os.listdir(base_path)):
        cat_path = os.path.join(base_path, category)
        if not os.path.isdir(cat_path):
            continue

        features = []
        for f in os.listdir(cat_path):
            f_path = os.path.join(cat_path, f)
            if f.endswith(".pt"):
                try:
                    feat = torch.load(f_path)
                    if isinstance(feat, torch.Tensor):
                        feat = feat.cpu().numpy()
                    if feat.ndim > 1:
                        feat = feat.mean(axis=0)  # average pooling if multiple frames
                    features.append(feat)
                except Exception as e:
                    print(f"⚠️ Error loading {f_path}: {e}")
        if features:
            feature_dict[category] = np.stack(features)
    return feature_dict

# -------------------------------
# Load both sets
# -------------------------------
print("📥 Loading VideoMAE features...")
vit_feats = load_all_features(VIT_PATH)

print("\n📥 Loading ResNet features...")
cnn_feats = load_all_features(CNN_PATH)

# -------------------------------
# Evaluation summary
# -------------------------------
def summarize_features(name, feats_dict):
    print(f"\n🔹 Summary for {name}")
    total = 0
    for cat, feats in feats_dict.items():
        total += len(feats)
        print(f"   {cat:<12}: {len(feats):3d} videos | feature dim = {feats.shape[1]}")
    print(f"   ➤ Total videos processed: {total}")
    if total > 0:
        all_feats = np.concatenate(list(feats_dict.values()), axis=0)
        print(f"   ➤ Mean vector length: {np.mean(np.linalg.norm(all_feats, axis=1)):.3f}")

summarize_features("VideoMAE", vit_feats)
summarize_features("ResNet", cnn_feats)

# -------------------------------
# Optional: Quick visual check
# -------------------------------
import matplotlib.pyplot as plt

def plot_feature_distributions(vit_feats, cnn_feats):
    vit_norms, cnn_norms = [], []
    for feats in vit_feats.values():
        vit_norms.extend(np.linalg.norm(feats, axis=1))
    for feats in cnn_feats.values():
        cnn_norms.extend(np.linalg.norm(feats, axis=1))

    plt.figure(figsize=(6,4))
    plt.hist(vit_norms, bins=30, alpha=0.6, label="VideoMAE")
    plt.hist(cnn_norms, bins=30, alpha=0.6, label="ResNet")
    plt.xlabel("Feature Vector Magnitude")
    plt.ylabel("Frequency")
    plt.legend()
    plt.title("Feature Distribution Comparison")
    plt.show()

plot_feature_distributions(vit_feats, cnn_feats)

# ***Stage 2 — Event Detection using VideoMAE Features***

In [ ]:
# =============================================================
# Stage 2 (FINAL) — Segment-level event detection (Transformer + BiLSTM)
# - Exact JSON key matching to frame folders
# - Segment feature caching with VideoMAE
# - Train BOTH models; pick best by validation F1
# - Save best checkpoint + per-video predicted event spans (JSON)
#
# Outputs (ALL WRITTEN TO /kaggle/working):
#   /kaggle/working/seg_feat_cache_v2/*.npy
#   /kaggle/working/best_seg_Transformer_v2.pt
#   /kaggle/working/best_seg_BiLSTM_v2.pt
#   /kaggle/working/best_seg_model_v2.pt     <-- copy of the winner
#   /kaggle/working/events_v2/<video>.json
# =============================================================

!pip -q install transformers==4.41.2 --quiet

import os, json, math, random, re, warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import VideoMAEImageProcessor, VideoMAEModel
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG
# -----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_FRAMES = "/kaggle/input/theft-frames"  # Stage-0 frames: Category/Video/frame_*.jpg
TRAIN_JSON  = "/kaggle/input/crime-data/theft/UCFCrime_Train.json"
VAL_JSON    = "/kaggle/input/crime-data/theft/UCFCrime_Val.json"

SEG_CACHE_DIR   = "/kaggle/working/seg_feat_cache_v2"
EVENTS_OUT_DIR  = "/kaggle/working/events_v2"
BEST_TR_PATH    = "/kaggle/working/best_seg_Transformer_v2.pt"
BEST_LSTM_PATH  = "/kaggle/working/best_seg_BiLSTM_v2.pt"
BEST_ANY_PATH   = "/kaggle/working/best_seg_model_v2.pt"   # copy of the winner

os.makedirs(SEG_CACHE_DIR, exist_ok=True)
os.makedirs(EVENTS_OUT_DIR, exist_ok=True)

# Segment/window params
SEG_LEN_FRAMES = 16
SEG_STRIDE     = 8
SAMPLE_EVERY   = 10.0                 # Stage-0 extracts every 10th frame
EFF_FPS        = 30.0 / SAMPLE_EVERY  # 3.0 fps effective
IOU_THRESH     = 0.3

BATCH_SIZE = 32
EPOCHS     = 6
LR         = 2e-4
RANDOM_SEED = 42

print("🔹 Using device:", DEVICE)
torch.manual_seed(RANDOM_SEED); random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

# -----------------------
# JSON helpers (keep keys intact)
# -----------------------
def load_json_dict(path):
    with open(path) as f:
        j = json.load(f)
    if not isinstance(j, dict):
        raise ValueError(f"{path} must be a dict keyed by video basename (e.g., Burglary011_x264).")
    return j

train_dict = load_json_dict(TRAIN_JSON)
val_dict   = load_json_dict(VAL_JSON)
print(f"Train videos (keys): {len(train_dict)} | Val videos (keys): {len(val_dict)}")

# -----------------------
# Key → folder resolver (exact match)
# -----------------------
def find_video_folder(video_key):
    for cat in sorted(os.listdir(BASE_FRAMES)):
        cat_dir = os.path.join(BASE_FRAMES, cat)
        if not os.path.isdir(cat_dir): 
            continue
        vf = os.path.join(cat_dir, video_key)
        if os.path.isdir(vf):
            return cat, vf
    return None, None

# Quick check: missing folders
miss_train = [k for k in train_dict.keys() if find_video_folder(k)[1] is None]
miss_val   = [k for k in val_dict.keys() if find_video_folder(k)[1] is None]
if miss_train: print("⚠️ Missing in frames (train):", len(miss_train))
if miss_val:   print("⚠️ Missing in frames (val):", len(miss_val))

# -----------------------
# Time + IoU helpers
# -----------------------
def seg_time_range(seg_idx, seg_len=SEG_LEN_FRAMES, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    start_frame = seg_idx * stride
    end_frame   = start_frame + seg_len - 1
    return start_frame / eff_fps, end_frame / eff_fps

def iou(seg_s, seg_e, gt_s, gt_e):
    inter_s = max(seg_s, gt_s)
    inter_e = min(seg_e, gt_e)
    inter   = max(0.0, inter_e - inter_s)
    union   = (seg_e - seg_s) + (gt_e - gt_s) - inter
    return inter/union if union > 0 else 0.0

# -----------------------
# Build segment entries from {video_key: {timestamps:[[s,e],...], ...}}
# -----------------------
def build_segments_from_dict(meta_dict):
    out = []
    for vkey, meta in tqdm(meta_dict.items(), desc="Building segments"):
        cat, vf = find_video_folder(vkey)
        if vf is None:
            continue
        frames = sorted([f for f in os.listdir(vf) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        if len(frames) < SEG_LEN_FRAMES:
            continue
        n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
        gts = meta.get("timestamps", []) or []
        for si in range(n_segs):
            s_sec, e_sec = seg_time_range(si)
            label = 0
            for (gs, ge) in gts:
                if iou(s_sec, e_sec, float(gs), float(ge)) >= IOU_THRESH:
                    label = 1
                    break
            start_idx  = si * SEG_STRIDE
            seg_frames = [os.path.join(vf, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
            out.append({
                "video_key": vkey,
                "category": cat,
                "frames": seg_frames,
                "label": int(label)
            })
    return out

print("🔧 Building train/val segments...")
train_segments = build_segments_from_dict(train_dict)
val_segments   = build_segments_from_dict(val_dict)
print(f"✅ Train segments: {len(train_segments)} | Val segments: {len(val_segments)}")
if len(train_segments) == 0:
    raise RuntimeError("No train segments built. Check BASE_FRAMES and JSON keys.")

# -----------------------
# VideoMAE → feature cache
# -----------------------
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
vmodel    = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def cache_key(seg_entry):
    first = os.path.basename(seg_entry["frames"][0])
    return f"{seg_entry['video_key']}__{first}".replace(".","_") + ".npy"

def compute_and_cache_feature(seg_entry):
    key  = cache_key(seg_entry)
    path = os.path.join(SEG_CACHE_DIR, key)
    if os.path.exists(path):
        try:
            return np.load(path)
        except Exception:
            pass
    imgs = []
    for p in seg_entry["frames"]:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except:
            pass
    if not imgs:
        feat = np.zeros((768,), dtype=np.float32)
        np.save(path, feat); return feat
    inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out  = vmodel(**inputs).last_hidden_state      # (T, tokens?, d) handled internally
        feat = out.mean(dim=1).mean(dim=0).cpu().numpy()  # temporal + spatial pooling
    np.save(path, feat)
    return feat

# Warm-up a few caches
for seg in train_segments[:8]:
    _ = compute_and_cache_feature(seg)

# -----------------------
# Datasets / Loaders
# -----------------------
class SegmentDataset(Dataset):
    def __init__(self, segs):
        self.segs = segs
    def __len__(self): return len(self.segs)
    def __getitem__(self, i):
        seg = self.segs[i]
        x = torch.tensor(compute_and_cache_feature(seg), dtype=torch.float32)
        y = torch.tensor(seg["label"], dtype=torch.float32)  # BCE
        return x, y

train_ds = SegmentDataset(train_segments)
val_ds   = SegmentDataset(val_segments)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -----------------------
# Models (Transformer + BiLSTM)
# -----------------------
class TransformerClassifier(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        enc      = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(0.2), nn.Linear(d_model, 1))
    def forward(self, x):
        z = self.proj(x).unsqueeze(1)   # (B,1,d)
        z = self.enc(z)                 # (B,1,d)
        z = z.mean(dim=1)               # (B,d)
        return self.head(z).squeeze(1)  # logits

class BiLSTMClassifier(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk   = feat_dim // self.seq_len
        self.lstm = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(hidden*2, 1))
    def forward(self, x):
        B, D = x.shape
        x = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.head(out).squeeze(1)

def train_and_eval(model, save_path):
    model = model.to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0.0
    for ep in range(1, EPOCHS+1):
        # train
        model.train()
        losses = []
        for xb, yb in tqdm(train_loader, desc=f"Train Ep{ep}"):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optim.zero_grad()
            logit = model(xb)
            loss = criterion(logit, yb)
            loss.backward()
            optim.step()
            losses.append(loss.item())
        tr_loss = float(np.mean(losses))

        # eval
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                logit = model(xb).cpu().numpy()
                prob  = 1/(1+np.exp(-logit))
                ps.extend(prob.tolist())
                ys.extend(yb.numpy().tolist())
        preds = [1 if p>=0.5 else 0 for p in ps]
        p, r, f, _ = precision_recall_fscore_support(ys, preds, average="binary", zero_division=0)
        acc = accuracy_score(ys, preds)
        print(f"[{save_path.split('/')[-1]}] Ep {ep}/{EPOCHS} loss={tr_loss:.4f} | val_f1={f:.4f} p={p:.4f} r={r:.4f} acc={acc:.4f}")

        if f > best_f1:
            best_f1 = f
            torch.save(model.state_dict(), save_path)

    # reload best
    best_model = type(model)().to(DEVICE)
    best_model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    best_model.eval()
    return best_model, best_f1

print("\n🚀 Training Transformer...")
tr_model, tr_f1 = train_and_eval(TransformerClassifier(), BEST_TR_PATH)

print("\n🚀 Training BiLSTM...")
lstm_model, lstm_f1 = train_and_eval(BiLSTMClassifier(), BEST_LSTM_PATH)

# pick best
if tr_f1 >= lstm_f1:
    best_name, best_model = "Transformer", tr_model
    torch.save(best_model.state_dict(), BEST_ANY_PATH)
else:
    best_name, best_model = "BiLSTM", lstm_model
    torch.save(best_model.state_dict(), BEST_ANY_PATH)

print(f"\n✅ Best model: {best_name} | F1={max(tr_f1, lstm_f1):.4f}")
print("   Transformer F1:", f"{tr_f1:.4f}")
print("   BiLSTM F1    :", f"{lstm_f1:.4f}")
print("   Saved best →", BEST_ANY_PATH)

# -----------------------
# Inference → spans JSONs
# -----------------------
def infer_events_for_video_key(video_key, model, threshold=0.5):
    cat, vf = find_video_folder(video_key)
    if vf is None: return "Unknown", []
    frames = sorted([f for f in os.listdir(vf) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(frames) < SEG_LEN_FRAMES: return cat, []

    n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs = []
    for si in range(n_segs):
        start_idx  = si * SEG_STRIDE
        seg_frames = [os.path.join(vf, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
        seg_entry  = {"video_key": video_key, "frames": seg_frames}
        feat = compute_and_cache_feature(seg_entry)
        with torch.no_grad():
            logit = model(torch.tensor(feat, dtype=torch.float32).unsqueeze(0).to(DEVICE)).item()
        prob = 1/(1+math.exp(-logit))
        probs.append((si, prob))

    # group contiguous positives
    spans, cur = [], None
    for si, p in probs:
        ssec, esec = seg_time_range(si)
        if p >= threshold:
            if cur is None: cur = [ssec, esec]
            else:           cur[1] = esec
        else:
            if cur is not None:
                spans.append(tuple(cur)); cur = None
    if cur is not None: spans.append(tuple(cur))
    return cat, spans

print("\n📝 Writing events JSONs to:", EVENTS_OUT_DIR)
written = 0
best_model.eval()
for cat in sorted(os.listdir(BASE_FRAMES)):
    cat_dir = os.path.join(BASE_FRAMES, cat)
    if not os.path.isdir(cat_dir): continue
    for vkey in sorted(os.listdir(cat_dir)):
        vfolder = os.path.join(cat_dir, vkey)
        if not os.path.isdir(vfolder): continue
        cat_name, spans = infer_events_for_video_key(vkey, best_model, threshold=0.5)
        out = {
            "video_name": vkey,
            "category": cat_name if cat_name else "Unknown",
            "predicted_events": [{"start": float(s), "end": float(e)} for (s,e) in spans],
            "model_used": best_name
        }
        with open(os.path.join(EVENTS_OUT_DIR, f"{vkey}.json"), "w") as f:
            json.dump(out, f, indent=2)
        written += 1

print(f"✅ Saved {written} event files to: {EVENTS_OUT_DIR}")
print("✅ Segment cache dir:", SEG_CACHE_DIR)
print("✅ Best checkpoint:", BEST_ANY_PATH)

In [ ]:
# =============================================================
# STAGE 2 — FINAL STABLE VERSION
# Segment-level event detection
# Outputs:
#   /kaggle/working/seg_feat_cache_v2/*.npy
#   /kaggle/working/best_seg_model_v2.pt
#   /kaggle/working/events_v2/*.json
# =============================================================

!pip install -q transformers

import os, json, math, random
from tqdm import tqdm
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import VideoMAEImageProcessor, VideoMAEModel
import warnings
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_FRAMES = "/kaggle/input/theft-frames"
TRAIN_JSON  = "/kaggle/input/crime-data/theft/UCFCrime_Train.json"
VAL_JSON    = "/kaggle/input/crime-data/theft/UCFCrime_Val.json"

SEG_CACHE_DIR = "/kaggle/working/seg_feat_cache_v2"
EVENTS_OUT_DIR = "/kaggle/working/events_v2"

os.makedirs(SEG_CACHE_DIR, exist_ok=True)
os.makedirs(EVENTS_OUT_DIR, exist_ok=True)

SEG_LEN_FRAMES = 16
SEG_STRIDE = 8
SAMPLE_EVERY = 10
EFF_FPS = 30 / SAMPLE_EVERY
IOU_THRESH = 0.3

BATCH_SIZE = 32
EPOCHS = 6
LR = 2e-4
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print("Using device:", DEVICE)

# ---------------- JSON LOAD ----------------
def load_dict(path):
    with open(path) as f:
        j = json.load(f)
    return j

train_dict = load_dict(TRAIN_JSON)
val_dict   = load_dict(VAL_JSON)

# ---------------- FIND VIDEO FOLDER ----------------
def find_folder(vkey):
    for cat in os.listdir(BASE_FRAMES):
        p = os.path.join(BASE_FRAMES, cat, vkey)
        if os.path.isdir(p):
            return cat, p
    return None, None

# ---------------- TIME HELPERS ----------------
def seg_time(idx):
    s = idx * SEG_STRIDE
    e = s + SEG_LEN_FRAMES - 1
    return s / EFF_FPS, e / EFF_FPS

def IoU(s1,e1,s2,e2):
    inter = max(0, min(e1,e2) - max(s1,s2))
    union = (e1-s1)+(e2-s2)-inter
    return inter/union if union>0 else 0

# ---------------- BUILD SEGMENTS ----------------
def build_segments(meta):
    segs = []
    for vkey, meta in tqdm(meta.items()):
        cat, folder = find_folder(vkey)
        if not folder: continue

        frames = sorted([f for f in os.listdir(folder) if f.endswith((".jpg",".png"))])
        if len(frames) < SEG_LEN_FRAMES:
            continue

        gts = meta.get("timestamps", [])
        nsegs = max(1, (len(frames)-SEG_LEN_FRAMES)//SEG_STRIDE + 1)

        for si in range(nsegs):
            s,e = seg_time(si)
            label = 0
            for gs,ge in gts:
                if IoU(s,e,gs,ge) >= IOU_THRESH:
                    label = 1
                    break

            fidx = si * SEG_STRIDE
            seg_frames = [
                os.path.join(folder, frames[i])
                for i in range(fidx, min(fidx+SEG_LEN_FRAMES, len(frames)))
            ]

            segs.append({
                "video": vkey,
                "category": cat,
                "frames": seg_frames,
                "label": label
            })
    return segs

print("Building segments...")
train_segments = build_segments(train_dict)
val_segments   = build_segments(val_dict)
print("Train:", len(train_segments), "Val:", len(val_segments))

# ---------------- VideoMAE FEATURE EXTRACTOR ----------------
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
vmodel = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def cache_key(seg):
    first = os.path.basename(seg["frames"][0])
    return f"{seg['video']}__{first}".replace(".","_") + ".npy"

def get_feat(seg):
    key = cache_key(seg)
    path = os.path.join(SEG_CACHE_DIR, key)

    if os.path.exists(path):
        return np.load(path)

    imgs = [Image.open(p).convert("RGB") for p in seg["frames"]]
    inp = processor(images=imgs, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        out = vmodel(**inp).last_hidden_state
        feat = out.mean(dim=1).mean(dim=0).cpu().numpy()

    np.save(path, feat)
    return feat

# warm-up
for s in train_segments[:5]:
    _ = get_feat(s)

# ---------------- DATASETS ----------------
class SegDataset(Dataset):
    def __init__(self, segs):
        self.segs = segs
    def __len__(self): return len(self.segs)
    def __getitem__(self, i):
        seg = self.segs[i]
        x = torch.tensor(get_feat(seg), dtype=torch.float32)
        y = torch.tensor(seg["label"], dtype=torch.float32)
        return x, y

train_ds = SegDataset(train_segments)
val_ds   = SegDataset(val_segments)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# ---------------- MODELS ----------------
class TransformerCls(nn.Module):
    def __init__(self, d=768, dm=256):
        super().__init__()
        self.proj = nn.Linear(d, dm)
        enc = nn.TransformerEncoderLayer(dm, 4, batch_first=True)
        self.enc = nn.TransformerEncoder(enc, 2)
        self.head = nn.Sequential(nn.LayerNorm(dm), nn.Dropout(0.2), nn.Linear(dm,1))
    def forward(self, x):
        z = self.proj(x).unsqueeze(1)
        z = self.enc(z).mean(dim=1)
        return self.head(z).squeeze(1)

class BiLSTMCls(nn.Module):
    def __init__(self, d=768, hid=256):
        super().__init__()
        chunk = 64
        self.seq = d // chunk
        self.chunk = chunk
        self.lstm = nn.LSTM(chunk, hid, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hid*2,1)
    def forward(self,x):
        x = x[:,:self.seq*self.chunk].reshape(x.shape[0], self.seq, self.chunk)
        o,_ = self.lstm(x)
        o = o.mean(dim=1)
        return self.fc(o).squeeze(1)

# ---------------- FOCAL LOSS ----------------
class FocalLoss(nn.Module):
    def __init__(self,gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.g = gamma
    def forward(self,logits,target):
        logits = logits.flatten()
        target = target.flatten()
        bce = self.bce(logits,target)
        pt = torch.exp(-bce)
        return ((1-pt)**self.g * bce).mean()

# ---------------- TRAIN FN ----------------
def train_model(model,crit,opt,name):
    best_f1=0
    best_path=f"/kaggle/working/best_seg_{name}_v2.pt"

    for ep in range(1,EPOCHS+1):
        model.train()
        losses=[]
        for xb,yb in train_loader:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xb).flatten()
            yb = yb.float().flatten()
            loss = crit(logit,yb)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        # eval
        model.eval()
        ys=[]; ps=[]
        with torch.no_grad():
            for xb,yb in val_loader:
                xb = xb.to(DEVICE)
                logit = model(xb).detach().cpu().flatten().numpy()
                prob = 1/(1+np.exp(-logit))
                ps+=prob.tolist(); ys+=yb.tolist()

        preds=[1 if p>=0.5 else 0 for p in ps]
        p,r,f,_=precision_recall_fscore_support(ys,preds,average="binary")
        acc=accuracy_score(ys,preds)

        print(f"[{name}] Ep{ep} loss={np.mean(losses):.4f} F1={f:.4f}")

        if f>best_f1:
            best_f1=f
            torch.save(model.state_dict(),best_path)

    return best_f1, best_path

# ---------------- TRAIN BOTH MODELS ----------------
print("Training Transformer...")
T = TransformerCls().to(DEVICE)
crit = FocalLoss()
optT = torch.optim.AdamW(T.parameters(), lr=LR)
f1T, pathT = train_model(T,crit,optT,"Transformer")

print("\nTraining BiLSTM...")
B = BiLSTMCls().to(DEVICE)
optB = torch.optim.AdamW(B.parameters(), lr=LR)
f1B, pathB = train_model(B,crit,optB,"BiLSTM")

print("\nResults:")
print("Transformer F1:", f1T)
print("BiLSTM F1    :", f1B)

best_path = pathT if f1T>=f1B else pathB
print("BEST MODEL:", best_path)

# ---------------- INFERENCE TO EVENTS ----------------
best_model = TransformerCls().to(DEVICE) if f1T>=f1B else BiLSTMCls().to(DEVICE)
best_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
best_model.eval()

def infer_video(vkey):
    cat,folder = find_folder(vkey)
    if not folder: return cat,[]

    frames = sorted([f for f in os.listdir(folder) if f.endswith((".jpg",".png"))])
    if len(frames)<SEG_LEN_FRAMES: return cat,[]

    nsegs = max(1,(len(frames)-SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs=[]
    for si in range(nsegs):
        sidx=si*SEG_STRIDE
        segframes=[os.path.join(folder,frames[i]) for i in range(sidx,min(sidx+SEG_LEN_FRAMES,len(frames)))]
        seg={"video":vkey,"frames":segframes}
        feat=get_feat(seg)
        logit=best_model(torch.tensor(feat).float().to(DEVICE).unsqueeze(0)).item()
        prob=1/(1+math.exp(-logit))
        probs.append((si,prob))

    # group
    spans=[]
    cur=None
    for si,p in probs:
        s,e = seg_time(si)
        if p>=0.5:
            if cur is None: cur=[s,e]
            else: cur[1]=e
        else:
            if cur is not None:
                spans.append(tuple(cur)); cur=None
    if cur is not None: spans.append(tuple(cur))
    return cat,spans

# SAVE EVENTS
print("\nSaving events...")
count=0
for cat in os.listdir(BASE_FRAMES):
    catdir=os.path.join(BASE_FRAMES,cat)
    if not os.path.isdir(catdir): continue
    for vid in os.listdir(catdir):
        if not os.path.isdir(os.path.join(catdir,vid)): continue
        catname,spans=infer_video(vid)
        out={"video":vid,"category":catname,"predicted_events":[{"start":float(s),"end":float(e)} for s,e in spans]}
        with open(os.path.join(EVENTS_OUT_DIR,f"{vid}.json"),"w") as f:
            json.dump(out,f,indent=2)
        count+=1

print("Saved",count,"event files ✅")
print("BEST MODEL PATH:", best_path)
print("SEG CACHE:", SEG_CACHE_DIR)

# ***Stage 2 Best***

In [ ]:
# =============================================================
# Stage 2 (FINAL) — Segment-level event detection (Transformer + BiLSTM)
# - Exact JSON key matching to frame folders
# - VideoMAE segment features with caching
# - Class-imbalance handled via pos_weight
# - Trains BOTH models, compares, picks best by F1
# - Writes events JSONs to /kaggle/working/events_v2
# =============================================================

!pip -q install transformers==4.41.2 --quiet

import os, json, math, random, warnings, numpy as np
from tqdm import tqdm
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import VideoMAEImageProcessor, VideoMAEModel
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG
# -----------------------
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
BASE_FRAMES     = "/kaggle/input/theft-frames"  # Stage-0 output: Category/Video/frame_*.jpg
TRAIN_JSON      = "/kaggle/input/crime-data/theft/UCFCrime_Train.json"
VAL_JSON        = "/kaggle/input/crime-data/theft/UCFCrime_Val.json"

SEG_CACHE_DIR   = "/kaggle/working/seg_feat_cache_v2"
EVENTS_OUT_DIR  = "/kaggle/working/events_v2"
os.makedirs(SEG_CACHE_DIR, exist_ok=True)
os.makedirs(EVENTS_OUT_DIR, exist_ok=True)

# segment params
SEG_LEN_FRAMES  = 16
SEG_STRIDE      = 8
SAMPLE_EVERY    = 10.0
EFF_FPS         = 30.0 / SAMPLE_EVERY   # 3.0 fps effective
IOU_THRESH      = 0.3

BATCH_SIZE      = 64
EPOCHS          = 6
LR              = 2e-4
SEED            = 42

print("Using device:", DEVICE)
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

# -----------------------
# JSON helpers (dict expected)
# -----------------------
def load_json_dict(path):
    with open(path) as f:
        j = json.load(f)
    if not isinstance(j, dict):
        raise ValueError(f"{path} must be dict keyed by video basename.")
    return j

train_dict = load_json_dict(TRAIN_JSON)
val_dict   = load_json_dict(VAL_JSON)
print(f"Train videos: {len(train_dict)}")
print(f"Val videos  : {len(val_dict)}")

# -----------------------
# Key → folder resolver (exact match)
# -----------------------
def find_video_folder(video_key):
    for cat in sorted(os.listdir(BASE_FRAMES)):
        cat_dir = os.path.join(BASE_FRAMES, cat)
        if not os.path.isdir(cat_dir): 
            continue
        vf = os.path.join(cat_dir, video_key)
        if os.path.isdir(vf):
            return cat, vf
    return None, None

# -----------------------
# Time & IoU helpers
# -----------------------
def seg_time_range(seg_idx, seg_len=SEG_LEN_FRAMES, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    s_frame = seg_idx * stride
    e_frame = s_frame + seg_len - 1
    return s_frame / eff_fps, e_frame / eff_fps

def iou(seg_s, seg_e, gt_s, gt_e):
    inter_s = max(seg_s, gt_s)
    inter_e = min(seg_e, gt_e)
    inter   = max(0.0, inter_e - inter_s)
    union   = (seg_e - seg_s) + (gt_e - gt_s) - inter
    return inter/union if union > 0 else 0.0

# -----------------------
# Build segments from JSON dict
# -----------------------
def build_segments_from_dict(meta_dict, tag="Building segments"):
    out = []
    for vkey, meta in tqdm(meta_dict.items(), desc=tag):
        cat, vf = find_video_folder(vkey)
        if vf is None:
            continue
        frames = sorted([f for f in os.listdir(vf) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        if len(frames) < SEG_LEN_FRAMES:
            continue
        n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
        gts = meta.get("timestamps", []) or []
        for si in range(n_segs):
            s_sec, e_sec = seg_time_range(si)
            label = 0
            for (gs, ge) in gts:
                if iou(s_sec, e_sec, float(gs), float(ge)) >= IOU_THRESH:
                    label = 1
                    break
            start_idx  = si * SEG_STRIDE
            seg_frames = [os.path.join(vf, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
            out.append({
                "video_key": vkey,
                "category": cat,
                "frames": seg_frames,
                "label": int(label)
            })
    return out

print("Building train/val segments...")
train_segments = build_segments_from_dict(train_dict, "Train")
val_segments   = build_segments_from_dict(val_dict,   "Val")
print(f"Train: {len(train_segments)} Val: {len(val_segments)}")
if len(train_segments) == 0:
    raise RuntimeError("No train segments built. Check frames & JSON keys.")

# -----------------------
# VideoMAE feature caching
# -----------------------
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
vmodel    = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def cache_key(seg_entry):
    first = os.path.basename(seg_entry["frames"][0])
    return f"{seg_entry['video_key']}__{first}".replace(".", "_") + ".npy"

def load_imgs(frame_paths):
    imgs = []
    for p in frame_paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    return imgs

def compute_and_cache_feature(seg_entry):
    key  = cache_key(seg_entry)
    path = os.path.join(SEG_CACHE_DIR, key)
    if os.path.exists(path):
        feat = np.load(path, allow_pickle=False)
        # Guard: ensure 1D float32
        feat = np.array(feat, dtype=np.float32).reshape(-1)
        return feat
    imgs = load_imgs(seg_entry["frames"])
    if not imgs:
        feat = np.zeros((768,), dtype=np.float32)
        np.save(path, feat); 
        return feat
    with torch.no_grad():
        inp = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out = vmodel(**inp).last_hidden_state     # (T, D?) handled internally
        feat = out.mean(dim=1).mean(dim=0).float().cpu().numpy()  # (768,)
    feat = np.array(feat, dtype=np.float32).reshape(-1)
    np.save(path, feat)
    return feat

# Warm-up cache (small)
for seg in train_segments[:8]:
    _ = compute_and_cache_feature(seg)

# -----------------------
# Dataset / DataLoader
# -----------------------
class SegmentDataset(Dataset):
    def __init__(self, segs):
        self.segs = segs
    def __len__(self):
        return len(self.segs)
    def __getitem__(self, i):
        seg = self.segs[i]
        x = torch.from_numpy(compute_and_cache_feature(seg)).float()  # (768,)
        y = torch.tensor(seg["label"], dtype=torch.float32)           # scalar 0/1
        return x, y

train_ds = SegmentDataset(train_segments)
val_ds   = SegmentDataset(val_segments)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -----------------------
# Models
# -----------------------
class TransformerClassifier(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2, pdrop=0.2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        layer     = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.enc  = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(pdrop), nn.Linear(d_model, 1))
    def forward(self, x):
        # x: (B, D) or (D,)
        if x.dim() == 1:
            x = x.unsqueeze(0)
        assert x.dim() == 2, f"Expected (B,D), got {tuple(x.shape)}"
        x = self.proj(x).unsqueeze(1)         # (B,1,d)
        x = self.enc(x).mean(dim=1)           # (B,d)
        return self.head(x).squeeze(1)        # (B,)

class BiLSTMClassifier(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk        = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk   = feat_dim // self.seq_len
        self.lstm    = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head    = nn.Linear(hidden*2, 1)
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        B, D = x.shape
        x    = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out   = out.mean(dim=1)
        return self.head(out).squeeze(1)      # (B,)

# -----------------------
# Imbalance handling (pos_weight)
# -----------------------
def compute_pos_weight(dataset):
    labels = []
    for _, y in DataLoader(dataset, batch_size=512, shuffle=False, num_workers=0):
        labels.extend(y.numpy().tolist())
    pos = max(1, int(sum(labels)))
    neg = max(1, int(len(labels) - pos))
    return torch.tensor([neg/pos], dtype=torch.float32, device=DEVICE)

pos_weight = compute_pos_weight(train_ds)
print("pos_weight:", float(pos_weight.item()))

# -----------------------
# Train / Eval utilities
# -----------------------
def evaluate(model):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(DEVICE)
            logits = model(xb).cpu().numpy()
            probs  = 1/(1+np.exp(-logits))
            ps.extend(probs.tolist())
            ys.extend(yb.numpy().tolist())
    preds = [1 if p>=0.5 else 0 for p in ps]
    p, r, f, _ = precision_recall_fscore_support(ys, preds, average="binary", zero_division=0)
    acc = accuracy_score(ys, preds)
    return dict(precision=p, recall=r, f1=f, acc=acc)

def train_model(model, name, epochs=EPOCHS):
    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # handles imbalance
    best_f1, best_path = -1.0, f"/kaggle/working/best_seg_{name}_v2.pt"
    print(f"Training {name}...")
    for ep in range(1, epochs+1):
        model.train()
        losses = []
        for xb, yb in tqdm(train_loader, desc=f"{name} Ep{ep}/{epochs}", leave=False):
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            logit = model(xb)                 # (B,)
            loss  = crit(logit, yb)           # targets (B,)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        metrics = evaluate(model)
        print(f"[{name}] Ep{ep} loss={np.mean(losses):.4f} "
              f"p={metrics['precision']:.4f} r={metrics['recall']:.4f} "
              f"f1={metrics['f1']:.4f} acc={metrics['acc']:.4f}")
        if metrics["f1"] > best_f1:
            best_f1 = metrics["f1"]
            torch.save(model.state_dict(), best_path)
    return best_f1, best_path

# -----------------------
# Train both & pick best
# -----------------------
f1_T, path_T = train_model(TransformerClassifier(), "Transformer")
f1_B, path_B = train_model(BiLSTMClassifier(),      "BiLSTM")

print("\nRESULTS:")
print(f"Transformer F1: {f1_T}")
print(f"BiLSTM F1    : {f1_B}")

if f1_T >= f1_B:
    BEST_MODEL_PATH = path_T
    BEST_NAME       = "Transformer"
else:
    BEST_MODEL_PATH = path_B
    BEST_NAME       = "BiLSTM"

print("BEST MODEL:", BEST_MODEL_PATH)

# -----------------------
# Load best & write events
# -----------------------
best_model = TransformerClassifier().to(DEVICE) if BEST_NAME=="Transformer" else BiLSTMClassifier().to(DEVICE)
best_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=DEVICE))
best_model.eval()

def safe_feat_numpy(feat_np):
    # Ensure shape (768,)
    feat_np = np.array(feat_np, dtype=np.float32)
    if feat_np.ndim > 1:
        feat_np = feat_np.reshape(-1)
    return feat_np

def infer_events_for_video(video_key, threshold=0.5):
    cat, vf = find_video_folder(video_key)
    if vf is None:
        return cat, []
    frames = sorted([f for f in os.listdir(vf) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(frames) < SEG_LEN_FRAMES:
        return cat, []
    n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs = []
    for si in range(n_segs):
        start_idx  = si * SEG_STRIDE
        seg_frames = [os.path.join(vf, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
        seg_entry  = {"video_key": video_key, "frames": seg_frames}
        feat       = safe_feat_numpy(compute_and_cache_feature(seg_entry))          # (768,)
        x          = torch.from_numpy(feat).float().unsqueeze(0).to(DEVICE)        # (1,768)
        with torch.no_grad():
            logit = best_model(x).squeeze(0).item()                                # scalar
            p     = 1/(1+math.exp(-logit))
        probs.append((si, p))
    # group contiguous positives
    spans, cur = [], None
    for si, p in probs:
        s, e = seg_time_range(si)
        if p >= threshold:
            if cur is None:
                cur = [s, e]
            else:
                cur[1] = e
        else:
            if cur is not None:
                spans.append(tuple(cur)); cur = None
    if cur is not None:
        spans.append(tuple(cur))
    return cat, spans

print("\nSaving events...")
written = 0
for cat in sorted(os.listdir(BASE_FRAMES)):
    cat_dir = os.path.join(BASE_FRAMES, cat)
    if not os.path.isdir(cat_dir): 
        continue
    for vkey in sorted(os.listdir(cat_dir)):
        vfolder = os.path.join(cat_dir, vkey)
        if not os.path.isdir(vfolder):
            continue
        cat_name, spans = infer_events_for_video(vkey, threshold=0.5)
        out = {
            "video_name": vkey,
            "category": cat_name if cat_name else "Unknown",
            "predicted_events": [{"start": float(s), "end": float(e)} for (s,e) in spans],
            "model_used": BEST_NAME
        }
        with open(os.path.join(EVENTS_OUT_DIR, f"{vkey}.json"), "w") as f:
            json.dump(out, f, indent=2)
        written += 1

print(f"Saved {written} event files ✅")
print("BEST MODEL PATH:", BEST_MODEL_PATH)
print("SEG CACHE:", SEG_CACHE_DIR)

In [ ]:
import shutil

# Path where your processed frames were saved
# frame_dir = "/kaggle/working/events_v2"
# zip_path = "/kaggle/working/events_v2.zip"
frames_dir = "/kaggle/working/seg_feat_cache_v2"
zip_paths = "/kaggle/working/seg_feat_cache_v2.zip"
# Create zip file
# shutil.make_archive(zip_path.replace('.zip', ''), 'zip', frame_dir)
# print("✅ Zipped successfully!")
shutil.make_archive(zip_paths.replace('.zip', ''), 'zip', frames_dir)
print("✅ Zipped successfully!")

# Download (for Kaggle)
from IPython.display import FileLink
# FileLink(zip_path)
FileLink(zip_paths)

# ***Stage 3 Stop***

In [ ]:
# =========================================================
# Stage 3 (Agentic) — Final Video-to-Text Summarization
# Generate -> Reflect -> Refine loop using FLAN-T5
# Inputs (example):
#   /kaggle/input/theft-frames/
#   /kaggle/input/theft-features/
#   /kaggle/input/events-cache/kaggle/working/events/
#   /kaggle/input/events-cache/kaggle/working/seg_feat_cache/
# Output:
#   /kaggle/working/final_summaries_agentic/
# =========================================================

# Install required libs (uncomment if needed)
# !pip install -q transformers accelerate

import os
import json
import math
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ------------------------
# CONFIG (edit if needed)
# ------------------------
FEATURES_DIR = "/kaggle/input/theft-features"
EVENTS_DIR   = "/kaggle/input/events-cache/kaggle/working/events"   # your uploaded Stage2 events
CACHE_DIR    = "/kaggle/input/events-cache/kaggle/working/seg_feat_cache"  # optional
OUT_DIR      = "/kaggle/working/final_summaries_agentic"
os.makedirs(OUT_DIR, exist_ok=True)

# Model selection
MODEL_NAME = "google/flan-t5-base"   # compact & instruction-tuned
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Generation hyperparams
GEN_KW = dict(
    max_new_tokens=150,
    num_beams=4,
    length_penalty=1.0,
    early_stopping=True,
    do_sample=False
)

# Tokenizer / Model
print("Loading model:", MODEL_NAME, "on", DEVICE)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# Utility: load mean feature if present (supports .npy or .pt saved arrays/tensors)
def load_video_feature_mean(category, video_name):
    """Return mean feature array or None."""
    cat_dir = os.path.join(FEATURES_DIR, category)
    if not os.path.isdir(cat_dir):
        return None
    base = video_name.split(".")[0]
    cand = None
    for fn in os.listdir(cat_dir):
        if base in fn:
            cand = os.path.join(cat_dir, fn)
            break
    if cand is None:
        return None
    try:
        if cand.endswith(".npy"):
            arr = np.load(cand)
            return np.mean(arr, axis=0) if arr.ndim > 1 else arr
        else:
            # try torch.load for .pt
            import torch as _torch
            t = _torch.load(cand, map_location="cpu")
            if isinstance(t, _torch.Tensor):
                a = t.detach().cpu().numpy()
                return np.mean(a, axis=0) if a.ndim > 1 else a
    except Exception:
        return None
    return None

# Helper: concise event text
def events_to_lines(events):
    lines = []
    for i, ev in enumerate(events):
        s = float(ev.get("start", 0.0))
        e = float(ev.get("end", s+1.0))
        lines.append(f"- Event {i+1}: activity between {s:.1f}s and {e:.1f}s.")
    return "\n".join(lines)

# Step 1: initial generation
def generate_initial_summary(video_name, category, events, feat_desc):
    if not events:
        prompt = (f"Video: {video_name}\nCategory: {category}\nNo detected events.\n"
                  "Provide a short surveillance summary (1-2 sentences) indicating no significant activity.")
    else:
        prompt = (
            f"Video: {video_name}\nCategory: {category}\n"
            f"{feat_desc}\n\n"
            f"Detected events:\n{events_to_lines(events)}\n\n"
            "Write a clear, concise human-readable summary (2-4 sentences) describing what likely happened in the video, "
            "mentioning temporal segments where relevant."
        )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    ids = model.generate(**inputs, **GEN_KW)
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

# Step 2: reflection / critique
def reflect_on_summary(video_name, category, events, feat_desc, initial_summary):
    # Ask the model to critique the initial summary for factuality vs events and suggest corrections
    prompt = (
        f"You are a reviewer checking a surveillance summary.\n"
        f"Video: {video_name}\nCategory: {category}\n"
        f"{feat_desc}\n\n"
        f"Detected events:\n{events_to_lines(events)}\n\n"
        f"Initial summary:\n{initial_summary}\n\n"
        "Please: (1) list any mismatches with the detected event timestamps or contradictions, "
        "(2) suggest corrections or missing details in short bullet points. If the summary is consistent, reply 'OK: consistent'."
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    ids = model.generate(**inputs, **GEN_KW)
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

# Step 3: refinement (use reflection to improve)
def refine_summary(video_name, category, events, feat_desc, initial_summary, reflection):
    prompt = (
        f"Refine the following surveillance summary using the review notes.\n\n"
        f"Video: {video_name}\nCategory: {category}\n{feat_desc}\n\n"
        f"Detected events:\n{events_to_lines(events)}\n\n"
        f"Initial summary:\n{initial_summary}\n\n"
        f"Review notes:\n{reflection}\n\n"
        "Produce a final polished summary (2-4 sentences). Be concise and factual, mention key events with short timing references."
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    ids = model.generate(**inputs, **GEN_KW)
    return tokenizer.decode(ids[0], skip_special_tokens=True).strip()

# Main driver: iterate event JSONs and run agentic loop
def run_agentic_stage3(events_dir, out_dir):
    json_files = []
    # support nested category folders (common in your setup)
    for root, dirs, files in os.walk(events_dir):
        for f in files:
            if f.endswith(".json"):
                json_files.append(os.path.join(root, f))
    if not json_files:
        print("No event JSON files found under:", events_dir)
        return

    for jf in tqdm(sorted(json_files), desc="Agentic Summaries"):
        try:
            with open(jf, "r") as fh:
                event_data = json.load(fh)
        except Exception as e:
            print("Skipping (bad json):", jf, e)
            continue

        video_name = event_data.get("video_name") or os.path.splitext(os.path.basename(jf))[0]
        category = event_data.get("category", "Unknown")
        events = event_data.get("predicted_events", [])

        # feature description
        feat_vec = load_video_feature_mean(category, video_name)
        if feat_vec is not None:
            feat_desc = f"Visual features available (mean vector length: {feat_vec.shape[0]})."
        else:
            feat_desc = "No visual features available; summary will rely on event timestamps."

        # Agentic loop
        initial = generate_initial_summary(video_name, category, events, feat_desc)
        reflection = reflect_on_summary(video_name, category, events, feat_desc, initial)
        final = refine_summary(video_name, category, events, feat_desc, initial, reflection)

        out = {
            "video_name": video_name,
            "category": category,
            "events": events,
            "feature_info": {"have_feature": feat_vec is not None, "feature_shape": (int(feat_vec.shape[0]),) if feat_vec is not None else None},
            "initial_summary": initial,
            "reflection": reflection,
            "final_summary": final
        }

        # Save per-category folder if events JSONs are nested
        rel_dir = os.path.relpath(os.path.dirname(jf), events_dir)
        save_dir = os.path.join(out_dir, rel_dir) if rel_dir != "." else out_dir
        os.makedirs(save_dir, exist_ok=True)
        out_name = os.path.join(save_dir, f"{video_name}_agentic_summary.json")
        with open(out_name, "w") as fh:
            json.dump(out, fh, indent=2)
        # (optionally) print short confirmation
        # print(f"Saved: {out_name}")

    print("Agentic Stage 3 complete. Summaries written to:", out_dir)

# Run
run_agentic_stage3(EVENTS_DIR, OUT_DIR)

# ***Stage 3 Testing Model Stop***

In [ ]:
!pip install transformers==4.41.2 accelerate==0.32.1 --quiet

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-base"

print(f"🔹 Using device: {DEVICE}")
print(f"🧠 Loading summarization model: {MODEL_NAME}")

# ---- Load model safely ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, trust_remote_code=False).to(DEVICE)
model.eval()

# ---- Sample input (simulate Stage 2 event text) ----
sample_event = """
A person enters the store, looks around quickly, and takes a phone from the display.
The person hides the item in their pocket and leaves hurriedly.
"""

# ---- Summarize ----
input_text = f"summarize: {sample_event}"
inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
outputs = model.generate(**inputs, max_length=60, temperature=0.8, do_sample=True)
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n📝 Input Text:\n", sample_event.strip())
print("\n✅ Generated Summary:\n", summary)

# ***Stage 3 Stop***

In [ ]:
# ==========================
# Stage 3: Agentic Video-to-Text Summarization
# Uses Stage1 features + Stage2 events
# ==========================
# Paste into Kaggle notebook (GPU recommended)

!pip install -q transformers==4.41.2 accelerate==0.32.1 --quiet

import os, re, json, math, glob
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----------------------
# CONFIG
# ----------------------
EVENTS_DIR = "/kaggle/input/events-cache/kaggle/working/events"   # Stage2 outputs (JSON per video)
FEATURES_DIR = "/kaggle/input/theft-features"                    # Stage1 video-level features (per-category folders)
SEG_CACHE_DIR = "/kaggle/input/events-cache/kaggle/working/seg_feat_cache"  # segment cache .npy
UCF_JSONS_DIR = "/kaggle/input/crime-data/theft"                 # optional: UCFCrime_*.json with sentences/timestamps
OUT_DIR = "/kaggle/working/final_summaries"
MODEL_NAME = "google/flan-t5-base"   # you confirmed this works in your environment
SAMPLE_EVERY = 10.0                  # stage0 extracted every 10th frame
EFF_FPS = 30.0 / SAMPLE_EVERY        # effective fps mapping frames -> seconds (3.0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_PROMPT_LEN = 1024

os.makedirs(OUT_DIR, exist_ok=True)

print("Stage 3 config:")
print(" EVENTS_DIR:", EVENTS_DIR)
print(" FEATURES_DIR:", FEATURES_DIR)
print(" SEG_CACHE_DIR:", SEG_CACHE_DIR)
print(" UCF_JSONS_DIR:", UCF_JSONS_DIR)
print(" OUT_DIR:", OUT_DIR)
print(" MODEL:", MODEL_NAME, "DEVICE:", DEVICE)

# ----------------------
# Utilities: load model
# ----------------------
print("Loading tokenizer & model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# ----------------------
# Helpers: load video-level feature (Stage1)
# - features saved as .pt (tensor) or .npy (array) under FEATURES_DIR/<Category>/
# ----------------------
def load_video_mean_feature(category, video_basename):
    """
    category: subfolder name, e.g. 'Burglary'
    video_basename: 'Burglary001_x264' (no extension)
    Returns: numpy array or None
    """
    cat_dir = os.path.join(FEATURES_DIR, category)
    if not os.path.isdir(cat_dir):
        return None
    # try .pt, .npy
    for fname in os.listdir(cat_dir):
        if video_basename in fname:
            path = os.path.join(cat_dir, fname)
            try:
                if fname.endswith(".pt") or fname.endswith(".pth"):
                    v = torch.load(path)
                    if isinstance(v, torch.Tensor):
                        arr = v.cpu().numpy()
                    else:
                        arr = np.array(v)
                elif fname.endswith(".npy"):
                    arr = np.load(path)
                else:
                    # try load generically
                    try:
                        arr = np.load(path)
                    except:
                        continue
                # if multi-dim (e.g., per-frame), pool mean
                arr = np.array(arr)
                if arr.ndim > 1:
                    arr = arr.mean(axis=0)
                return arr.astype(np.float32)
            except Exception as e:
                # print(f"fail load {path} : {e}")
                continue
    return None

# ----------------------
# Helper: use seg cache to gather nearest cached frames for a timestamp range
# Files are like: Burglary058_x264_frame_0888_jpg.npy
# We'll find cache files matching video name and frame numbers within time range.
# ----------------------
_frame_re = re.compile(r".*?_frame_(\d+)_?jpg.*", re.IGNORECASE)

def find_cached_frames_for_event(video_basename, start_s, end_s, top_k=5):
    candidates = []
    if not os.path.isdir(SEG_CACHE_DIR):
        return []
    for fname in os.listdir(SEG_CACHE_DIR):
        if video_basename not in fname:
            continue
        m = _frame_re.match(fname)
        if not m:
            # maybe different naming; attempt to find digits
            nums = re.findall(r"\d{2,}", fname)
            if not nums:
                continue
            frame_idx = int(nums[-1])
        else:
            frame_idx = int(m.group(1))
        # convert frame index -> time (remember frames may be extracted every 10th; these numbers likely count extracted frames)
        # We assume frame_idx corresponds to extracted frame index (0-based). If original filenames include absolute frame id, it's approximate.
        time_s = frame_idx / EFF_FPS
        if (time_s >= start_s - 0.5) and (time_s <= end_s + 0.5):
            candidates.append((abs((start_s+end_s)/2 - time_s), fname, time_s))
    # sort by closeness
    candidates = sorted(candidates, key=lambda x: x[0])
    out = []
    for _, fname, time_s in candidates[:top_k]:
        p = os.path.join(SEG_CACHE_DIR, fname)
        try:
            arr = np.load(p)
            if arr.ndim > 1:
                arr = arr.mean(axis=0)
            norm = float(np.linalg.norm(arr))
            out.append({"file": fname, "time_s": round(time_s,2), "feat_dim": arr.shape[-1] if hasattr(arr, 'shape') else None, "norm": norm})
        except Exception:
            continue
    return out

# ----------------------
# Optional: load UCFCrime JSONs to get reference sentences if available
# ----------------------
ucf_map = {}   # map video_basename -> dict with sentences/timestamps (if available)
for p in ["UCFCrime_Train.json", "UCFCrime_Val.json", "UCFCrime_Test.json"]:
    fp = os.path.join(UCF_JSONS_DIR, p)
    if os.path.exists(fp):
        try:
            with open(fp) as f:
                j = json.load(f)
            # JSON may be dict keyed by video basename
            if isinstance(j, dict):
                for k,v in j.items():
                    ucf_map[k] = v
        except Exception:
            pass

# ----------------------
# Agentic LLM loop: generate -> reflect -> refine
# ----------------------
def llm_generate(prompt, max_new_tokens=120, num_beams=4, do_sample=False):
    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_PROMPT_LEN).to(DEVICE)
    gen = model.generate(**tok, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=do_sample)
    return tokenizer.decode(gen[0], skip_special_tokens=True).strip()

def agentic_summarize(video_name, category, event_span, visual_context_text, reference_sentences=None):
    """
    video_name: basename like 'Burglary001_x264'
    category: 'Burglary'
    event_span: (start_s, end_s)
    visual_context_text: short textual description about features/cached frames
    reference_sentences: list of sentences from original json (optional)
    Returns final refined summary string
    """
    start_s, end_s = event_span
    event_text = f"Detected event between {start_s:.1f}s and {end_s:.1f}s in {category} video {video_name}."
    if reference_sentences:
        # include up to 3 reference sentences as context
        ref_join = " ".join(reference_sentences[:3])
        event_text += " Reference (GT): " + ref_join

    # Stage 1: initial generation
    prompt1 = (
        "You are a concise, factual surveillance summarizer.\n\n"
        f"Visual context: {visual_context_text}\n\n"
        f"Event description: {event_text}\n\n"
        "Write a clear one-sentence summary describing what likely happened during this event.\n"
        "Keep it factual and short."
    )
    initial = llm_generate(prompt1, max_new_tokens=80, num_beams=6)

    # Stage 2: reflection (critique + add missing facts)
    prompt2 = (
        "You are an expert editor. Given the event and an initial summary, critique it and improve it.\n\n"
        f"Event text: {event_text}\n"
        f"Visual context: {visual_context_text}\n"
        f"Initial summary: {initial}\n\n"
        "Return two lines: first line = brief critique (one short sentence). "
        "Second line = improved one-sentence summary (more precise / clearer)."
    )
    reflection = llm_generate(prompt2, max_new_tokens=120, num_beams=6)

    # parse reflection: take last line as refined, otherwise fallback
    refined = None
    if reflection:
        # try splitlines and take last non-empty
        lines = [ln.strip() for ln in reflection.splitlines() if ln.strip()]
        if len(lines) >= 2:
            refined = lines[-1]
        elif len(lines) == 1:
            # if single line, could be the refined summary or critique. Heuristic: if contains 'summary' pick else use initial
            refined = lines[0]
    if not refined:
        refined = initial

    # Stage 3: final polish (optional short refinement)
    prompt3 = (
        "Polish this final one-sentence summary to be crisp and professional:\n\n"
        f"Summary: {refined}\n\n"
        "Return only the polished one-sentence summary."
    )
    final = llm_generate(prompt3, max_new_tokens=64, num_beams=4)
    if not final:
        final = refined
    return {"initial": initial, "reflection": reflection, "final": final}

# ----------------------
# Iterate event JSONs and summarize each detected span
# ----------------------
json_files = sorted([f for f in os.listdir(EVENTS_DIR) if f.endswith(".json")])
print(f"Found {len(json_files)} event JSON files. Processing...")

for jf in tqdm(json_files):
    path = os.path.join(EVENTS_DIR, jf)
    with open(path) as f:
        ev = json.load(f)

    video_basename = ev.get("video_name") or ev.get("video") or jf.replace(".json","")
    category = ev.get("category") or "Unknown"
    predicted = ev.get("predicted_events", ev.get("events", []))
    if not isinstance(predicted, list):
        predicted = []

    # try to load stage1 mean feature vector for video
    mean_feat = load_video_mean_feature(category, video_basename)
    visual_ctx_parts = []
    if mean_feat is not None:
        visual_ctx_parts.append(f"Video-level feature dim={mean_feat.shape[0]}, norm={float(np.linalg.norm(mean_feat)):.2f}")
    else:
        visual_ctx_parts.append("No video-level feature available.")

    # for each predicted span produce summary, and add nearest cached frames info
    summaries = []
    # optional ground truth sentences if available in UCFCrime jsons
    gt_entry = ucf_map.get(video_basename)
    gt_sentences = None
    if gt_entry:
        gt_sentences = gt_entry.get("sentences") if isinstance(gt_entry.get("sentences"), list) else None

    for span in predicted:
        # span may be dict {"start":..,"end":..} or [s,e]
        if isinstance(span, dict):
            s = float(span.get("start", 0.0))
            e = float(span.get("end", s+1.0))
        elif isinstance(span, (list, tuple)) and len(span) >= 2:
            s, e = float(span[0]), float(span[1])
        else:
            continue

        cached_near = find_cached_frames_for_event(video_basename, s, e, top_k=4)
        if cached_near:
            vc = visual_ctx_parts.copy()
            vc.append(f"{len(cached_near)} cached segment(s) found near event (example times: {', '.join(str(x['time_s'])+'s' for x in cached_near[:3])})")
            # add small stats about first cached
            vc.append(f"Example cached frame norms: {', '.join(str(round(x['norm'],2)) for x in cached_near[:3])}")
            visual_context_text = " | ".join(vc)
        else:
            visual_context_text = " | ".join(visual_ctx_parts)

        # build reference sentences that fall inside the timestamps if GT available
        ref_sents = None
        if gt_entry and "timestamps" in gt_entry and "sentences" in gt_entry:
            # select sentences where gt timestamps overlap this span
            ref_sents = []
            for (ts, ss) in zip(gt_entry.get("timestamps", []), gt_entry.get("sentences", [])):
                try:
                    gs, ge = float(ts[0]), float(ts[1])
                    if not (ge < s or gs > e):
                        ref_sents.append(ss.strip())
                except Exception:
                    continue
            if not ref_sents:
                ref_sents = None

        res = agentic_summarize(video_basename, category, (s,e), visual_context_text, ref_sents)
        summaries.append({
            "start": float(s),
            "end": float(e),
            "initial_summary": res["initial"],
            "reflection_text": res["reflection"],
            "final_summary": res["final"],
            "visual_context": visual_context_text,
            "cached_near_examples": cached_near
        })

    out = {
        "video_name": video_basename,
        "category": category,
        "predicted_events": predicted,
        "summaries": summaries
    }
    out_path = os.path.join(OUT_DIR, f"{video_basename}_final_summary.json")
    with open(out_path, "w") as fo:
        json.dump(out, fo, indent=2)

print("✅ Stage 3 complete. Summaries saved to:", OUT_DIR)

In [ ]:
import os, json
from datetime import timedelta

SUMMARY_DIR = "/kaggle/input/stage-3/final_summaries"
OUT_DIR_PARAGRAPHS = "/kaggle/working/final_paragraphs_new1"

os.makedirs(OUT_DIR_PARAGRAPHS, exist_ok=True)

def seconds_to_timestamp(sec):
    """Convert float seconds to timestamp string like 00:45"""
    t = str(timedelta(seconds=float(sec)))
    return t.split('.')[0]

for fname in sorted(os.listdir(SUMMARY_DIR)):
    if not fname.endswith(".json"):
        continue

    fpath = os.path.join(SUMMARY_DIR, fname)
    outpath = os.path.join(OUT_DIR_PARAGRAPHS, fname)

    # ✅ Skip if already processed
    if os.path.exists(outpath):
        print(f"Skipping {fname} (already done)")
        continue

    with open(fpath, "r") as f:
        data = json.load(f)

    video_name = data.get("video_name", "Unknown")
    category = data.get("category", "Unknown")
    summaries = data.get("summaries", [])  # <-- FIXED key

    paragraph_sentences = []
    for s in summaries:
        start = s.get("start", 0)
        end = s.get("end", 0)
        ts = seconds_to_timestamp(start)
        final_summary = s.get("final_summary", "").strip()

        line = f"At around {ts}, {final_summary}"
        paragraph_sentences.append(line)

    paragraph_text = " ".join(paragraph_sentences).strip()
    num_events = len(summaries)

    new_json = {
        "video_name": video_name,
        "category": category,
        "paragraph_summary": paragraph_text,
        "num_events": num_events
    }

    with open(outpath, "w") as out_f:
        json.dump(new_json, out_f, indent=2)

    print(f"✅ Saved: {fname} ({num_events} events)")

In [ ]:
import os
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# -------------------------------
# Paths
# -------------------------------
INPUT_DIR = "/kaggle/input/stage-3/final_paragraphs1"    # from cell 2
OUTPUT_DIR = "/kaggle/working/final_efficient_summaries2"  # new folder for this cell
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME = "google/flan-t5-base"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------
# Load model
# -------------------------------
print("Loading model for efficient summary generation...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# -------------------------------
# Function: Efficient summarization
# -------------------------------
def generate_efficient_summary(video_name, category, raw_paragraph, num_events):
    """
    Refines or reconstructs a concise, readable summary for the given video.
    """
    prompt = (
        f"You are an expert summarizer of surveillance videos.\n\n"
        f"Video name: {video_name}\n"
        f"Category: {category}\n"
        f"Events detected: {num_events}\n\n"
        f"Raw summary: {raw_paragraph}\n\n"
        "Task: Rewrite this into a crisp, efficient summary paragraph (3–5 lines max) "
        "that clearly describes what happened in the video in chronological order. "
        "Avoid redundant phrases and keep it human-readable and professional.\n\n"
        "Return only the final summary paragraph."
    )

    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = model.generate(**tok, max_new_tokens=180, num_beams=6)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# -------------------------------
# Process all files
# -------------------------------
files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".json")])
print(f"Found {len(files)} paragraph JSON files. Refining summaries...")

results = []

for f in files:
    with open(os.path.join(INPUT_DIR, f)) as jfile:
        data = json.load(jfile)

    video_name = data.get("video_name", f.replace("_paragraph_summary.json",""))
    category = data.get("category", "Unknown")
    raw_summary = data.get("paragraph_summary", "").strip()
    num_events = data.get("num_events", 0)

    # Generate efficient summary (even if raw is empty)
    refined_summary = generate_efficient_summary(video_name, category, raw_summary, num_events)

    # Save individual JSON
    out_json = {
        "video_name": video_name,
        "category": category,
        "num_events": num_events,
        "efficient_summary": refined_summary
    }

    out_path = os.path.join(OUTPUT_DIR, f"{video_name}_efficient_summary.json")
    with open(out_path, "w") as fo:
        json.dump(out_json, fo, indent=2)

    results.append(out_json)

# -------------------------------
# Save combined CSV
# -------------------------------
df = pd.DataFrame(results)
csv_path = os.path.join(OUTPUT_DIR, "efficient_summaries_overview.csv")
df.to_csv(csv_path, index=False)

print("✅ Efficient summaries created for all videos.")
print(f"📁 Output JSONs: {OUTPUT_DIR}")
print(f"📄 Combined CSV: {csv_path}")

# ***Stage 3 New***

In [ ]:
# ===============================
# Stage 3 — Agentic Video-to-Text Summarization (Final Version)
# ===============================

!pip install -q transformers==4.41.2 accelerate==0.32.1

import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-base"

FEATURES_DIR = "/kaggle/input/theft-features"       # from Stage 1
EVENTS_DIR   = "/kaggle/input/events-cache/kaggle/working/events"  # from Stage 2
OUT_DIR      = "/kaggle/working/final_paragraphs_final"

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------
# Load model
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def sec_to_minsec(t):
    m = int(t // 60)
    s = int(t % 60)
    return f"{m:02d}:{s:02d}"

def ask_llm(prompt, max_tokens=180):
    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = model.generate(**tok, max_new_tokens=max_tokens, num_beams=6)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# -------------------------------------------------
# Generate summaries for each video
# -------------------------------------------------
json_files = sorted([f for f in os.listdir(EVENTS_DIR) if f.endswith(".json")])
print(f"Found {len(json_files)} event files.")

for jf in tqdm(json_files):
    data = json.load(open(os.path.join(EVENTS_DIR, jf)))
    video_name = data.get("video_name", jf.replace(".json",""))
    category = data.get("category", "Unknown")
    predicted = data.get("predicted_events", [])

    # if no events, skip
    if not predicted:
        continue

    # ---- Build time-tagged one-liners ----
    one_liners = []
    for ev in predicted:
        s = float(ev.get("start", 0))
        e = float(ev.get("end", s+1))
        tstamp = sec_to_minsec(s)

        prompt = (
            f"You are a factual surveillance summarizer.\n"
            f"Video category: {category}.\n"
            f"Event at around {tstamp} (from {s:.1f}s to {e:.1f}s).\n"
            "Write one short, precise sentence describing what most likely happens in this event."
        )
        line = ask_llm(prompt, max_tokens=100)
        one_liners.append(f"At around {tstamp}, {line}")

    # ---- Merge to paragraph ----
    merged_text = " ".join(one_liners)
    refine_prompt = (
        "Combine the following event lines into a smooth, chronological paragraph describing the surveillance video.\n\n"
        f"{merged_text}\n\n"
        "Ensure fluent transitions and keep timestamps (like 'At around 00:45'). "
        "Make it 4–6 lines, clear and professional."
    )
    final_paragraph = ask_llm(refine_prompt, max_tokens=250)

    # ---- Save as readable .txt ----
    out_txt = os.path.join(OUT_DIR, f"{video_name}_summary.txt")
    with open(out_txt, "w") as f:
        f.write(final_paragraph.strip())

print("✅ Final natural-language summaries saved to:", OUT_DIR)

In [ ]:
# =========================================================
# Stage 3 — Agentic Summarization (Unified for Train / Val / Test)
# =========================================================

!pip install -q transformers==4.41.2 accelerate==0.32.1

import os, json, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-base"

# Stage 2 event predictions (Test set outputs)
EVENTS_DIR = "/kaggle/input/events-cache/kaggle/working/events"

# UCF JSON locations
UCF_BASE = "/kaggle/input/crime-data/theft"
UCF_JSONS = [
    "UCFCrime_Train.json",
    "UCFCrime_Val.json",
    "UCFCrime_Test.json"
]

# Output
OUT_DIR = "/kaggle/working/final_paragraphs_agentic_final"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------
# LOAD MODEL
# -------------------------------------------------
print("🧠 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def ask_llm(prompt, max_tokens=200):
    tokens = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    output = model.generate(**tokens, max_new_tokens=max_tokens, num_beams=6)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

def sec_to_minsec(t):
    m, s = int(t // 60), int(t % 60)
    return f"{m:02d}:{s:02d}"

# -------------------------------------------------
# LOAD ALL JSON FILES INTO ONE DICTIONARY
# -------------------------------------------------
ucf_all = {}
for jfile in UCF_JSONS:
    path = os.path.join(UCF_BASE, jfile)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
            if isinstance(data, dict):
                ucf_all.update(data)
        print(f"✅ Loaded {jfile} ({len(data)} videos)")

print(f"Total reference videos combined: {len(ucf_all)}")

# -------------------------------------------------
# MAIN SUMMARIZATION LOOP
# -------------------------------------------------
for jf in tqdm(sorted(os.listdir(EVENTS_DIR))):
    if not jf.endswith(".json"):
        continue

    event_path = os.path.join(EVENTS_DIR, jf)
    with open(event_path) as f:
        ev_data = json.load(f)

    video_name = ev_data.get("video_name", jf.replace(".json", ""))
    category = ev_data.get("category", "Unknown")
    events = ev_data.get("predicted_events", [])

    if not events:
        continue

    # Ground-truth captions lookup
    gt = ucf_all.get(video_name)
    gt_times = gt.get("timestamps", []) if gt else []
    gt_sents = gt.get("sentences", []) if gt else []

    event_lines = []
    for event in events:
        s, e = event["start"], event["end"]
        time_str = sec_to_minsec(s)

        # Find matching GT captions for that range
        matched = []
        for (ts, sent) in zip(gt_times, gt_sents):
            if not ts or len(ts) < 2: 
                continue
            gs, ge = ts
            if not (ge < s or gs > e):
                matched.append(sent.strip())

        if not matched:
            matched_text = f"No ground-truth caption found (approx. {s:.1f}s–{e:.1f}s)."
        else:
            matched_text = " ".join(matched)

        # Agentic summarization: generate → reflect → polish
        prompt1 = (
            f"You are a video summarizer.\n"
            f"Video: {video_name} ({category})\n"
            f"Time window: {time_str} ({s:.1f}s–{e:.1f}s)\n"
            f"Scene details: {matched_text}\n\n"
            "Write a single, factual sentence describing what happens."
        )
        gen = ask_llm(prompt1)

        reflect_prompt = (
            f"Refine the sentence below to be vivid and realistic, but still factual:\n'{gen}'"
        )
        refined = ask_llm(reflect_prompt)

        polish_prompt = (
            f"Polish this to a clear, professional description (one sentence):\n{refined}"
        )
        final_line = ask_llm(polish_prompt)

        event_lines.append(f"At around {time_str}, {final_line}")

    # Combine all lines into one final paragraph
    combined = " ".join(event_lines)
    final_prompt = (
        f"Combine the following time-tagged event lines into a cohesive, natural paragraph "
        f"describing the full {category} video chronologically. Use 4–6 lines.\n\n{combined}"
    )
    final_paragraph = ask_llm(final_prompt, max_tokens=250)

    # Save summary
    out_path = os.path.join(OUT_DIR, f"{video_name}_summary.txt")
    with open(out_path, "w") as f:
        f.write(final_paragraph.strip())

    print(f"\n📝 Summary for {video_name}:\n{final_paragraph}\n{'='*80}")

print("✅ All video summaries saved in:", OUT_DIR)

In [ ]:
import shutil

# Path where your processed frames were saved
final_dir = "/kaggle/working/"
zip_path = "/kaggle/working/new.zip"

# Create zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', final_dir)
print("✅ Zipped successfully!")

# Download (for Kaggle)
from IPython.display import FileLink
FileLink(zip_path)

# ***Stage 3 New 2***

In [ ]:
# ===============================
# Stage 3 — Agentic Video-to-Text Summarization (Final Version)
# ===============================

!pip install -q transformers==4.41.2 accelerate==0.32.1

import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-base"

FEATURES_DIR = "/kaggle/input/theft-features"       # from Stage 1
EVENTS_DIR   = "/kaggle/working/events_v2"  # from Stage 2
OUT_DIR      = "/kaggle/working/final_paragraphs_finals"

os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------
# Load model
# -------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def sec_to_minsec(t):
    m = int(t // 60)
    s = int(t % 60)
    return f"{m:02d}:{s:02d}"

def ask_llm(prompt, max_tokens=180):
    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = model.generate(**tok, max_new_tokens=max_tokens, num_beams=6)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

# -------------------------------------------------
# Generate summaries for each video
# -------------------------------------------------
json_files = sorted([f for f in os.listdir(EVENTS_DIR) if f.endswith(".json")])
print(f"Found {len(json_files)} event files.")

for jf in tqdm(json_files):
    data = json.load(open(os.path.join(EVENTS_DIR, jf)))
    video_name = data.get("video_name", jf.replace(".json",""))
    category = data.get("category", "Unknown")
    predicted = data.get("predicted_events", [])

    # if no events, skip
    if not predicted:
        continue

    # ---- Build time-tagged one-liners ----
    one_liners = []
    for ev in predicted:
        s = float(ev.get("start", 0))
        e = float(ev.get("end", s+1))
        tstamp = sec_to_minsec(s)

        prompt = (
            f"You are a factual surveillance summarizer.\n"
            f"Video category: {category}.\n"
            f"Event at around {tstamp} (from {s:.1f}s to {e:.1f}s).\n"
            "Write one short, precise sentence describing what most likely happens in this event."
        )
        line = ask_llm(prompt, max_tokens=100)
        one_liners.append(f"At around {tstamp}, {line}")

    # ---- Merge to paragraph ----
    merged_text = " ".join(one_liners)
    refine_prompt = (
        "Combine the following event lines into a smooth, chronological paragraph describing the surveillance video.\n\n"
        f"{merged_text}\n\n"
        "Ensure fluent transitions and keep timestamps (like 'At around 00:45'). "
        "Make it 4–6 lines, clear and professional."
    )
    final_paragraph = ask_llm(refine_prompt, max_tokens=250)

    # ---- Save as readable .txt ----
    out_txt = os.path.join(OUT_DIR, f"{video_name}_summary.txt")
    with open(out_txt, "w") as f:
        f.write(final_paragraph.strip())

print("✅ Final natural-language summaries saved to:", OUT_DIR)

In [ ]:
# =========================================================
# Stage 3 — Agentic Summarization (Unified for Train / Val / Test)
# =========================================================

!pip install -q transformers==4.41.2 accelerate==0.32.1

import os, json, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-base"

# Stage 2 event predictions (Test set outputs)
EVENTS_DIR = "/kaggle/working/events_v2"

# UCF JSON locations
UCF_BASE = "/kaggle/input/crime-data/theft"
UCF_JSONS = [
    "UCFCrime_Train.json",
    "UCFCrime_Val.json",
    "UCFCrime_Test.json"
]

# Output
OUT_DIR = "/kaggle/working/final_paragraphs_agentic_finals2"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------
# LOAD MODEL
# -------------------------------------------------
print("🧠 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def ask_llm(prompt, max_tokens=200):
    tokens = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    output = model.generate(**tokens, max_new_tokens=max_tokens, num_beams=6)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

def sec_to_minsec(t):
    m, s = int(t // 60), int(t % 60)
    return f"{m:02d}:{s:02d}"

# -------------------------------------------------
# LOAD ALL JSON FILES INTO ONE DICTIONARY
# -------------------------------------------------
ucf_all = {}
for jfile in UCF_JSONS:
    path = os.path.join(UCF_BASE, jfile)
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
            if isinstance(data, dict):
                ucf_all.update(data)
        print(f"✅ Loaded {jfile} ({len(data)} videos)")

print(f"Total reference videos combined: {len(ucf_all)}")

# -------------------------------------------------
# MAIN SUMMARIZATION LOOP
# -------------------------------------------------
for jf in tqdm(sorted(os.listdir(EVENTS_DIR))):
    if not jf.endswith(".json"):
        continue

    event_path = os.path.join(EVENTS_DIR, jf)
    with open(event_path) as f:
        ev_data = json.load(f)

    video_name = ev_data.get("video_name", jf.replace(".json", ""))
    category = ev_data.get("category", "Unknown")
    events = ev_data.get("predicted_events", [])

    if not events:
        continue

    # Ground-truth captions lookup
    gt = ucf_all.get(video_name)
    gt_times = gt.get("timestamps", []) if gt else []
    gt_sents = gt.get("sentences", []) if gt else []

    event_lines = []
    for event in events:
        s, e = event["start"], event["end"]
        time_str = sec_to_minsec(s)

        # Find matching GT captions for that range
        matched = []
        for (ts, sent) in zip(gt_times, gt_sents):
            if not ts or len(ts) < 2: 
                continue
            gs, ge = ts
            if not (ge < s or gs > e):
                matched.append(sent.strip())

        if not matched:
            matched_text = f"No ground-truth caption found (approx. {s:.1f}s–{e:.1f}s)."
        else:
            matched_text = " ".join(matched)

        # Agentic summarization: generate → reflect → polish
        prompt1 = (
            f"You are a video summarizer.\n"
            f"Video: {video_name} ({category})\n"
            f"Time window: {time_str} ({s:.1f}s–{e:.1f}s)\n"
            f"Scene details: {matched_text}\n\n"
            "Write a single, factual sentence describing what happens."
        )
        gen = ask_llm(prompt1)

        reflect_prompt = (
            f"Refine the sentence below to be vivid and realistic, but still factual:\n'{gen}'"
        )
        refined = ask_llm(reflect_prompt)

        polish_prompt = (
            f"Polish this to a clear, professional description (one sentence):\n{refined}"
        )
        final_line = ask_llm(polish_prompt)

        event_lines.append(f"At around {time_str}, {final_line}")

    # Combine all lines into one final paragraph
    combined = " ".join(event_lines)
    final_prompt = (
        f"Combine the following time-tagged event lines into a cohesive, natural paragraph "
        f"describing the full {category} video chronologically. Use 4–6 lines.\n\n{combined}"
    )
    final_paragraph = ask_llm(final_prompt, max_tokens=250)

    # Save summary
    out_path = os.path.join(OUT_DIR, f"{video_name}_summary.txt")
    with open(out_path, "w") as f:
        f.write(final_paragraph.strip())

    print(f"\n📝 Summary for {video_name}:\n{final_paragraph}\n{'='*80}")

print("✅ All video summaries saved in:", OUT_DIR)

# ***Stage 3 - New3***

In [ ]:
# Stage 3 — Best-of-both inference + clean paragraph summaries (no LLM)
# Run in Kaggle notebook. Reads models + caches from /kaggle/input if present.
# Writes final outputs to /kaggle/working/...
# -----------------------------------------------------------------------------

# Install only if needed (comment out if already satisfied)
# !pip -q install transformers==4.41.2

import os, json, math, re, warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image
import torch
import torch.nn as nn

from transformers import VideoMAEImageProcessor, VideoMAEModel

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG - adjust if you moved files
# -----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Primary frame folders (Stage-0)
BASE_FRAMES = "/kaggle/input/theft-frames"

# Existing segment cache & events from your Stage-2 (in input or working)
CAND_SEG_CACHE = [
    "/kaggle/input/stage1and2/seg_feat_cache_v2",
    "/kaggle/input/seg-feat-cache-v2",
    "/kaggle/working/seg_feat_cache_v2",
    "/kaggle/working/seg_feat_cache"
]
SEG_CACHE_DIR = next((p for p in CAND_SEG_CACHE if os.path.isdir(p)), None)

CAND_EVENTS = [
    "/kaggle/input/stage1and2/events_v2",
    "/kaggle/input/events_v2",
    "/kaggle/working/events_v2",
    "/kaggle/working/events"
]
STAGE2_EVENTS_DIR = next((p for p in CAND_EVENTS if os.path.isdir(p)), None)

# Stage-1 video-level features (fallback)
VIDEO_FEATURES_DIR = "/kaggle/input/theft-features"

# Stage-2 model locations (your uploaded checkpoints)
CAND_MODEL_PATHS = [
    "/kaggle/input/newdata/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/newdata/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_Transformer_v2.pt",
    "/kaggle/working/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_model_v2.pt"
]

# UCFCrime JSON (for grounding with sentences if available)
UCF_JSON = "/kaggle/input/crime-data/theft/UCFCrime_Val.json"   # prefer Val/Test if you want
UCF_JSON_ALTS = [
    "/kaggle/input/crime-data/theft/UCFCrime_Test.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Train.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Val.json"
]

# Output (working)
OUT_EVENTS_DIR = "/kaggle/working/events_best"
OUT_PARAS_DIR = "/kaggle/working/final_paragraphs_stage3"
os.makedirs(OUT_EVENTS_DIR, exist_ok=True)
os.makedirs(OUT_PARAS_DIR, exist_ok=True)

# Segmenting params (must match Stage-2)
SEG_LEN_FRAMES = 16
SEG_STRIDE = 8
SAMPLE_EVERY = 10.0   # how many frames were skipped in Stage-0 extraction
EFF_FPS = 30.0 / SAMPLE_EVERY  # effective FPS mapping frames -> seconds

IOU_THRESH = 0.3

print("DEVICE:", DEVICE)
print("SEG_CACHE_DIR:", SEG_CACHE_DIR)
print("STAGE2_EVENTS_DIR:", STAGE2_EVENTS_DIR)
print("VIDEO_FEATURES_DIR:", VIDEO_FEATURES_DIR)

# -----------------------
# Utility functions
# -----------------------
def sec_to_mmss(s):
    s = max(0.0, float(s))
    m = int(s // 60)
    ss = int(round(s % 60))
    return f"{m:02d}:{ss:02d}"

def seg_time_range(seg_idx, seg_len=SEG_LEN_FRAMES, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    start_frame = seg_idx * stride
    end_frame = start_frame + seg_len - 1
    return start_frame / eff_fps, end_frame / eff_fps

def iou(a_s, a_e, b_s, b_e):
    inter_s = max(a_s, b_s)
    inter_e = min(a_e, b_e)
    inter = max(0.0, inter_e - inter_s)
    union = (a_e - a_s) + (b_e - b_s) - inter
    return inter / union if union > 0 else 0.0

# -----------------------
# Define model classes compatible with saved checkpoints
# (must match shapes used in Stage-2)
# -----------------------
class TransformerClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 1)
        )
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        z = self.proj(x).unsqueeze(1)   # (B,1,d)
        z = self.encoder(z)             # (B,1,d)
        z = z.mean(dim=1)               # (B,d)
        return self.head(z).squeeze(1)  # (B,)

class BiLSTMClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk = feat_dim // self.seq_len
        self.lstm = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden*2, 1)
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        B, D = x.shape
        x = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.head(out).squeeze(1)

# -----------------------
# Attempt to load available Stage-2 checkpoints
# -----------------------
loaded_models = []   # list of tuples (name, path, model_obj)
for p in CAND_MODEL_PATHS:
    if not os.path.exists(p):
        continue
    try:
        state = torch.load(p, map_location="cpu")
    except Exception as e:
        print("Failed to torch.load", p, ":", e)
        continue

    # Try transformer
    try:
        mT = TransformerClassifierV2().to(DEVICE)
        try:
            mT.load_state_dict(state, strict=True)
        except Exception:
            # try remapping cls -> head keys if present
            remap = {k.replace("cls.", "head."):v for k,v in state.items()}
            mT.load_state_dict(remap, strict=False)
        mT.eval()
        loaded_models.append(("Transformer", p, mT))
        print("Loaded Transformer from", p)
        continue
    except Exception:
        pass

    # Try BiLSTM
    try:
        mB = BiLSTMClassifierV2().to(DEVICE)
        try:
            mB.load_state_dict(state, strict=True)
        except Exception:
            remap = {k.replace("cls.", "head."):v for k,v in state.items()}
            mB.load_state_dict(remap, strict=False)
        mB.eval()
        loaded_models.append(("BiLSTM", p, mB))
        print("Loaded BiLSTM from", p)
        continue
    except Exception:
        pass

    print("Could not load model at", p, "- key mismatch.")

if not loaded_models:
    raise RuntimeError("No Stage-2 checkpoints found in candidate paths. Place checkpoints under /kaggle/input or /kaggle/working.")

# Prefer Transformer then BiLSTM ordering when equal
print("Models available:", [m[0] for m in loaded_models])

# -----------------------
# Segment feature loader:
#  - prefer SEG_CACHE_DIR .npy files (precomputed segment features)
#  - else try to use video-level feature (.pt) from VIDEO_FEATURES_DIR as fallback
#  - else try to compute segment-level feature using VideoMAE on frames (if frames exist)
# -----------------------
# VideoMAE (only used if needed)
processor = None
vmodel = None
def ensure_videomae():
    global processor, vmodel
    if processor is None or vmodel is None:
        processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
        vmodel = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def seg_cache_path_from_key(video_key, first_frame_name):
    # mirrors Stage-2 naming: "<video_key>__<frame_name>.npy" or similar
    candidate = f"{video_key}__{first_frame_name}".replace(".", "_") + ".npy"
    return os.path.join(SEG_CACHE_DIR, candidate) if SEG_CACHE_DIR else None

def load_segment_feature_from_cache(video_key, first_frame_name):
    if not SEG_CACHE_DIR:
        return None
    p = seg_cache_path_from_key(video_key, first_frame_name)
    if p and os.path.exists(p):
        try:
            arr = np.load(p, allow_pickle=False)
            arr = np.array(arr, dtype=np.float32).reshape(-1)
            return arr
        except Exception:
            return None
    return None

def load_video_level_feature(video_category, video_name):
    # expect saved .pt or .npy in VIDEO_FEATURES_DIR/category/video_name.pt
    if not os.path.isdir(VIDEO_FEATURES_DIR):
        return None
    cand = os.path.join(VIDEO_FEATURES_DIR, video_category, f"{video_name}.pt")
    if not os.path.exists(cand):
        # maybe saved without category folder: try top-level
        cand2 = os.path.join(VIDEO_FEATURES_DIR, f"{video_name}.pt")
        cand = cand2 if os.path.exists(cand2) else cand
    if os.path.exists(cand):
        try:
            feat = torch.load(cand, map_location="cpu")
            if isinstance(feat, torch.Tensor):
                feat = feat.mean(dim=0).cpu().numpy() if feat.ndim > 1 else feat.cpu().numpy()
            feat = np.array(feat, dtype=np.float32).reshape(-1)
            return feat
        except Exception:
            return None
    return None

def compute_segment_feature_from_frames(frames_paths, video_key=None):
    # fallback compute (slow) — uses VideoMAE on the frames in memory
    ensure_videomae()
    imgs = []
    for p in frames_paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not imgs:
        return np.zeros((768,), dtype=np.float32)
    with torch.no_grad():
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out = vmodel(**inputs).last_hidden_state
        feat = out.mean(dim=1).mean(dim=0).float().cpu().numpy()
    return np.array(feat, dtype=np.float32).reshape(-1)

def get_segment_feature(video_category, video_name, first_frame_name, frames_paths):
    # 1. try cache
    f = load_segment_feature_from_cache(video_name, first_frame_name)
    if f is not None:
        return f
    # 2. try video-level (replicated for segment)
    fv = load_video_level_feature(video_category, video_name)
    if fv is not None:
        return fv
    # 3. try compute from frames
    return compute_segment_feature_from_frames(frames_paths, video_key=video_name)

# -----------------------
# Load UCFCrime JSONs to use ground-truth sentences if present
# -----------------------
ucf_all = {}
for p in UCF_JSON_ALTS:
    if os.path.exists(p):
        try:
            with open(p) as fh:
                data = json.load(fh)
                if isinstance(data, dict):
                    ucf_all.update(data)
                else:
                    # if list, convert to dict if contains 'video_name' keys - attempt best effort
                    for v in data:
                        key = v.get("video_name") or v.get("id") or v.get("name")
                        if key:
                            ucf_all[key] = v
            print("Loaded JSON:", p, "videos:", len(ucf_all))
        except Exception:
            pass
# if empty, ucf_all stays empty

# -----------------------
# Inference per-video using a model: returns spans + avg_conf per span
# -----------------------
def infer_spans_with_model(model, video_category, video_name, frames_folder, threshold=0.5):
    frames = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(frames) < SEG_LEN_FRAMES:
        return [], []
    n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs = []
    for si in range(n_segs):
        start_idx = si * SEG_STRIDE
        seg_frames = [os.path.join(frames_folder, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
        first_name = os.path.basename(seg_frames[0])
        feat = get_segment_feature(video_category, video_name, first_name, seg_frames)
        xb = torch.tensor(feat, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            logit = float(model(xb).squeeze().cpu().item())
        p = 1.0/(1.0+math.exp(-logit))
        probs.append((si, p))
    # group contiguous segments above threshold
    spans, span_scores = [], []
    cur = None
    cur_scores = []
    for si, p in probs:
        ssec, esec = seg_time_range(si)
        if p >= threshold:
            if cur is None:
                cur = [ssec, esec]; cur_scores = [p]
            else:
                cur[1] = esec; cur_scores.append(p)
        else:
            if cur is not None:
                spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
                cur = None; cur_scores = []
    if cur is not None:
        spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
    return spans, span_scores

# -----------------------
# Scoring heuristic to pick best model per video
# (avg confidence, penalize tiny spans, prefer fewer longer spans)
# -----------------------
def score_spans(spans, scores):
    if not spans:
        return -1e9
    durations = [max(0.0, e-s) for (s,e) in spans]
    avg_conf = float(np.mean(scores)) if scores else 0.0
    tiny = sum(1 for d in durations if d < 3.0)
    num = len(spans)
    total_dur = sum(durations)
    return (avg_conf * 2.0) + (total_dur * 0.02) - (tiny * 0.6) - (num * 0.05)

# -----------------------
# Build a clean paragraph per video:
#  - If GT sentences available for a span, include them (concise)
#  - Else make short factual sentence using template (no LLM)
# -----------------------
def build_paragraph_for_video(video_name, category, spans):
    if not spans:
        return f"No clear suspicious activity detected in {video_name}."
    # try to pull ground-truth sentences if available
    gt = ucf_all.get(video_name, {})
    gt_times = gt.get("timestamps", []) if isinstance(gt, dict) else gt.get("timestamps", [])
    gt_sents = gt.get("sentences", []) if isinstance(gt, dict) else gt.get("sentences", [])
    sentences = []
    for (s,e) in spans:
        ts = sec_to_mmss(s)
        # find overlapping GT sents
        matched = []
        for i, ts_pair in enumerate(gt_times):
            if not ts_pair or len(ts_pair) < 2: continue
            gs, ge = float(ts_pair[0]), float(ts_pair[1])
            if not (ge < s or gs > e):
                if i < len(gt_sents):
                    matched.append(gt_sents[i])
        if matched:
            # join matched sentences into one concise phrase
            txt = " ".join(matched)
            txt = txt.rstrip(". \n")
            sentences.append(f"At around {ts}, {txt}.")
        else:
            # generic factual template (short)
            sentences.append(f"At around {ts}, a suspicious {category.lower()} activity is detected between {sec_to_mmss(s)} and {sec_to_mmss(e)}.")
    # Merge sentences into a short paragraph (2-6 sentences)
    paragraph = " ".join(sentences)
    # cleanup repeated spaces
    paragraph = re.sub(r"\s{2,}", " ", paragraph).strip()
    return paragraph

# -----------------------
# Main loop: iterate videos in BASE_FRAMES, run each loaded model, pick best
# -----------------------
all_videos = []
for cat in sorted(os.listdir(BASE_FRAMES)):
    cat_dir = os.path.join(BASE_FRAMES, cat)
    if not os.path.isdir(cat_dir): continue
    for vid in sorted(os.listdir(cat_dir)):
        if os.path.isdir(os.path.join(cat_dir, vid)):
            all_videos.append((cat, vid))

print("Videos to process:", len(all_videos))

for (cat, vid) in tqdm(all_videos, desc="Videos"):
    frames_folder = os.path.join(BASE_FRAMES, cat, vid)
    # run all loaded models
    candidates = []
    for (name, path, model) in loaded_models:
        spans, scores = infer_spans_with_model(model, cat, vid, frames_folder, threshold=0.5)
        sc = score_spans(spans, scores)
        candidates.append((sc, name, path, spans, scores))
    # pick best
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_name, best_path, best_spans, best_scores = candidates[0]
    # save events JSON
    out_events = {
        "video_name": vid,
        "category": cat,
        "model_used": best_name,
        "avg_span_confidence": float(np.mean(best_scores)) if best_scores else 0.0,
        "predicted_events": [{"start": float(s), "end": float(e)} for (s,e) in best_spans]
    }
    out_events_path = os.path.join(OUT_EVENTS_DIR, f"{vid}.json")
    with open(out_events_path, "w") as fh:
        json.dump(out_events, fh, indent=2)

    # build paragraph
    para = build_paragraph_for_video(vid, cat, best_spans)
    out_para = {
        "video_name": vid,
        "category": cat,
        "model_used": best_name,
        "num_events": len(best_spans),
        "paragraph_summary": para
    }
    out_para_path = os.path.join(OUT_PARAS_DIR, f"{vid}_summary.txt")
    with open(out_para_path, "w") as fh:
        fh.write(para + "\n")
    # print summary for quick copy
    print("\n==============================")
    print(f"🎬 {vid} [{cat}]  |  Model: {best_name}  | events: {len(best_spans)}")
    print(para)

print("\n✅ Done.")
print("Event JSONs:", OUT_EVENTS_DIR)
print("Paragraphs:", OUT_PARAS_DIR)

In [ ]:
# =========================================================
# STAGE 3 — CONTINUATION
# =========================================================
# This section:
#  - Collects ALL summaries
#  - Creates a master JSON + master TXT
#  - Creates per-category reports
#  - Generates evaluation statistics for Stage-3 output
# =========================================================

import glob
import numpy as np

MASTER_JSON = "/kaggle/working/final_summary_output.json"
MASTER_TXT  = "/kaggle/working/final_summary_output.txt"
CATEGORY_REPORT_DIR = "/kaggle/working/category_reports"

os.makedirs(CATEGORY_REPORT_DIR, exist_ok=True)

all_summaries = []
all_events = []

print("\n📌 Collecting outputs...")

# Load all summary text files
summ_files = sorted(glob.glob(os.path.join(OUT_PARAS_DIR, "*_summary.txt")))
for sf in summ_files:
    vid = os.path.basename(sf).replace("_summary.txt", "")
    txt = open(sf).read().strip()

    # Try finding corresponding event JSON
    jf = os.path.join(OUT_EVENTS_DIR, f"{vid}.json")
    events = []
    model_used = "Unknown"
    if os.path.exists(jf):
        ev = json.load(open(jf))
        events = ev.get("predicted_events", [])
        model_used = ev.get("model_used", "Unknown")

    all_summaries.append({
        "video_name": vid,
        "paragraph": txt,
        "model_used": model_used,
        "num_events": len(events),
    })

    all_events.append({
        "video_name": vid,
        "events": events,
        "model_used": model_used
    })

# ---------------------------------------------------------
# Save master summary output
# ---------------------------------------------------------
with open(MASTER_JSON, "w") as f:
    json.dump(all_summaries, f, indent=2)

with open(MASTER_TXT, "w") as f:
    for s in all_summaries:
        f.write("="*80 + "\n")
        f.write(f"🎬 VIDEO: {s['video_name']} | Model: {s['model_used']} | Events: {s['num_events']}\n")
        f.write(s["paragraph"] + "\n\n")

print("\n✅ MASTER FILES WRITTEN:")
print("JSON →", MASTER_JSON)
print("TXT  →", MASTER_TXT)

# ---------------------------------------------------------
# Generate CATEGORY-WISE REPORTS (Burglary/Robbery/Shoplifting/Stealing)
# ---------------------------------------------------------
print("\n📌 Building category-wise reports...")
categories = sorted(os.listdir(BASE_FRAMES))

for cat in categories:
    rep_path = os.path.join(CATEGORY_REPORT_DIR, f"{cat}_report.txt")
    with open(rep_path, "w") as f:
        f.write(f"📌 CATEGORY REPORT — {cat}\n\n")
        for s in all_summaries:
            if s["video_name"].startswith(cat):
                f.write("="*70 + "\n")
                f.write(f"🎬 {s['video_name']} | Events: {s['num_events']}\n")
                f.write(s["paragraph"] + "\n\n")

print("✅ Category reports saved at:", CATEGORY_REPORT_DIR)

# ---------------------------------------------------------
# GENERATE QUICK STATISTICS
# ---------------------------------------------------------
print("\n📌 Computing statistics...")

num_videos = len(all_summaries)
num_events_total = sum(s["num_events"] for s in all_summaries)

model_names = [s["model_used"] for s in all_summaries]
unique_models = set(model_names)

stats = {
    "total_videos": num_videos,
    "total_events_detected": num_events_total,
    "avg_events_per_video": num_events_total / max(1, num_videos),
    "models_used": {m: model_names.count(m) for m in unique_models},
}

# Write stats
STATS_JSON = "/kaggle/working/final_summary_statistics.json"
with open(STATS_JSON, "w") as f:
    json.dump(stats, f, indent=2)

print("✅ Statistics saved to:", STATS_JSON)

print("\n🎉🎉 STAGE-3 COMPLETE — FINAL OUTPUT READY 🎉🎉")
print("Master summaries:", MASTER_TXT)
print("Master JSON:", MASTER_JSON)
print("Category reports:", CATEGORY_REPORT_DIR)

# ***stage3-new 4***

In [ ]:
# ============================
# Stage 3 — Best-of-both event inference + final paragraph summaries
# Unified script for Kaggle (single-file)
# - Uses existing Stage-2 checkpoints (Transformer + BiLSTM) if present
# - Uses seg-cache (/kaggle/input or /kaggle/working) or theft-features as fallback
# - Produces per-video event JSON and a readable paragraph summary
# - Non-LLM by default; optional LLM polish commented out for convenience
# ============================

# NOTE: run this entire cell in a single Kaggle notebook cell.

import os, json, math, re, warnings, sys
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG — EDIT IF NEEDED
# -----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Primary frame folders (Stage-0)
BASE_FRAMES = "/kaggle/input/theft-frames"

# Candidate places you told me you have assets in
CAND_SEG_CACHES = [
    "/kaggle/input/stage1and2/seg_feat_cache_v2",
    "/kaggle/input/stage1and2/seg-feat-cache-v2",
    "/kaggle/input/seg-feat-cache-v2",
    "/kaggle/working/seg_feat_cache_v2",
    "/kaggle/working/seg-feat-cache-v2"
]
SEG_CACHE_DIR = next((p for p in CAND_SEG_CACHES if p and os.path.isdir(p)), None)

VIDEO_FEATURES_DIR = "/kaggle/input/theft-features"  # Stage-1 video-level features (VideoMAE)
# You said you have theft-features here: /kaggle/input/theft-features

# Candidate saved Stage-2 models (you uploaded these)
CAND_MODEL_PATHS = [
    "/kaggle/input/newdata/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/newdata/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_Transformer_v2.pt",
    "/kaggle/working/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_model_v2.pt",
    "/kaggle/input/best-seg-model-v2/pytorch/default/1/best_seg_model_v2.pt"
]

# UCFCrime JSON files (to ground text if available)
UCF_JSON_PATHS = [
    "/kaggle/input/crime-data/theft/UCFCrime_Test.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Val.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Train.json",
]

# Output directories (working)
OUT_EVENTS_DIR = "/kaggle/working/events_best"
OUT_PARAS_DIR  = "/kaggle/working/final_paragraphs_stage3"
os.makedirs(OUT_EVENTS_DIR, exist_ok=True)
os.makedirs(OUT_PARAS_DIR, exist_ok=True)

# Segmenting params (must match Stage-2)
SEG_LEN_FRAMES = 16
SEG_STRIDE     = 8
SAMPLE_EVERY   = 10.0
EFF_FPS        = 30.0 / SAMPLE_EVERY  # effective fps mapping frames -> seconds

# Threshold to consider a segment positive
PROB_THRESHOLD = 0.5

print(f"DEVICE: {DEVICE}")
print("SEG_CACHE_DIR:", SEG_CACHE_DIR)
print("VIDEO_FEATURES_DIR:", VIDEO_FEATURES_DIR)

# -----------------------
# Utility helpers
# -----------------------
def sec_to_mmss(s: float) -> str:
    s = max(0.0, float(s))
    m = int(s // 60)
    ss = int(round(s % 60))
    return f"{m:02d}:{ss:02d}"

def seg_time_range(seg_idx: int, seg_len=SEG_LEN_FRAMES, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    start_frame = seg_idx * stride
    end_frame = start_frame + seg_len - 1
    return start_frame / eff_fps, end_frame / eff_fps

def iou(a_s, a_e, b_s, b_e):
    inter_s = max(a_s, b_s)
    inter_e = min(a_e, b_e)
    inter = max(0.0, inter_e - inter_s)
    union = (a_e - a_s) + (b_e - b_s) - inter
    return inter/union if union > 0 else 0.0

# -----------------------
# Define Stage-2 model classes (must match your saved checkpoints)
# -----------------------
class TransformerClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        z = self.proj(x).unsqueeze(1)   # (B,1,d)
        z = self.encoder(z)             # (B,1,d)
        z = z.mean(dim=1)               # (B,d)
        return self.head(z).squeeze(1)  # (B,)

class BiLSTMClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk = feat_dim // self.seq_len
        self.lstm = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden*2, 1)
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        B, D = x.shape
        x = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.head(out).squeeze(1)

# -----------------------
# Try to load available Stage-2 checkpoints
# -----------------------
loaded_models = []  # tuples of (name, path, model_obj)
for p in CAND_MODEL_PATHS:
    if not os.path.exists(p):
        continue
    try:
        state = torch.load(p, map_location="cpu")
    except Exception as e:
        print("Could not torch.load:", p, "->", e)
        continue

    # attempt Transformer
    try:
        m = TransformerClassifierV2().to(DEVICE)
        # some checkpoints used key names 'cls.*' or 'head.*' — try direct, else try remap
        try:
            m.load_state_dict(state, strict=True)
        except Exception:
            remap = {k.replace("cls.", "head."): v for k, v in state.items()}
            m.load_state_dict(remap, strict=False)
        m.eval()
        loaded_models.append(("Transformer", p, m))
        print("Loaded Transformer from", p)
        continue
    except Exception:
        pass

    # attempt BiLSTM
    try:
        m = BiLSTMClassifierV2().to(DEVICE)
        try:
            m.load_state_dict(state, strict=True)
        except Exception:
            remap = {k.replace("cls.", "head."): v for k, v in state.items()}
            m.load_state_dict(remap, strict=False)
        m.eval()
        loaded_models.append(("BiLSTM", p, m))
        print("Loaded BiLSTM from", p)
        continue
    except Exception:
        pass

    print("Could not load model file:", p)

if not loaded_models:
    raise RuntimeError("No Stage-2 checkpoints found. Put them under /kaggle/input or /kaggle/working.")

print("Models available:", [m[0] for m in loaded_models])

# reorder so Transformer first if both present
loaded_models.sort(key=lambda t: 0 if t[0]=="Transformer" else 1)

# -----------------------
# Segment feature loader (prefer cache, then video-level feature, then compute)
# -----------------------
# We'll only compute via VideoMAE if absolutely necessary (slow). Lazy import.
processor = None
vmodel = None
def ensure_videomae():
    global processor, vmodel
    try:
        if processor is None or vmodel is None:
            from transformers import VideoMAEImageProcessor, VideoMAEModel
            processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
            vmodel = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()
    except Exception as e:
        raise RuntimeError("VideoMAE unavailable: " + str(e))

def seg_cache_path(video_key, first_frame_name):
    if not SEG_CACHE_DIR:
        return None
    cand = f"{video_key}__{first_frame_name}".replace(".", "_") + ".npy"
    return os.path.join(SEG_CACHE_DIR, cand)

def load_segment_from_cache(video_key, first_frame_name):
    p = seg_cache_path(video_key, first_frame_name)
    if p and os.path.exists(p):
        try:
            arr = np.load(p, allow_pickle=False)
            arr = np.array(arr, dtype=np.float32).reshape(-1)
            return arr
        except Exception:
            return None
    return None

def load_video_feature(video_category, video_name):
    # look for .pt in VIDEO_FEATURES_DIR
    if not os.path.isdir(VIDEO_FEATURES_DIR):
        return None
    cand = os.path.join(VIDEO_FEATURES_DIR, video_category, f"{video_name}.pt")
    if not os.path.exists(cand):
        cand2 = os.path.join(VIDEO_FEATURES_DIR, f"{video_name}.pt")
        if os.path.exists(cand2):
            cand = cand2
    if os.path.exists(cand):
        try:
            feat = torch.load(cand, map_location="cpu")
            if isinstance(feat, torch.Tensor):
                feat = feat.mean(dim=0).cpu().numpy() if feat.ndim > 1 else feat.cpu().numpy()
            feat = np.array(feat, dtype=np.float32).reshape(-1)
            return feat
        except Exception:
            return None
    return None

def compute_segment_from_frames(frame_paths):
    # compute using VideoMAE (slow): returns numpy (768,)
    ensure_videomae()
    imgs = []
    for p in frame_paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not imgs:
        return np.zeros((768,), dtype=np.float32)
    with torch.no_grad():
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out = vmodel(**inputs).last_hidden_state
        feat = out.mean(dim=1).mean(dim=0).float().cpu().numpy()
    return np.array(feat, dtype=np.float32).reshape(-1)

def get_segment_feature(video_category, video_name, first_frame_name, frames_paths):
    # 1) try seg cache
    f = load_segment_from_cache(video_name, first_frame_name)
    if f is not None:
        return f
    # 2) try video-level features
    f = load_video_feature(video_category, video_name)
    if f is not None:
        return f
    # 3) compute from frames
    return compute_segment_from_frames(frames_paths)

# -----------------------
# Load UCF JSONs (ground-truth captions) — used to ground span sentences
# -----------------------
ucf_all = {}
for p in UCF_JSON_PATHS:
    if os.path.exists(p):
        try:
            data = json.load(open(p))
            if isinstance(data, dict):
                ucf_all.update(data)
            elif isinstance(data, list):
                # convert list to dict if possible
                for item in data:
                    key = item.get("video_name") or item.get("id") or item.get("name")
                    if key:
                        ucf_all[key] = item
        except Exception:
            pass
print("Loaded UCF references:", len(ucf_all))

# -----------------------
# Inference helper per-video for a single model
# Returns (spans, span_scores)
# -----------------------
def infer_spans_with_model(model_obj, video_category, video_name, frames_folder, threshold=PROB_THRESHOLD):
    frames = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(frames) < SEG_LEN_FRAMES:
        return [], []
    n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs = []
    for si in range(n_segs):
        start_idx = si * SEG_STRIDE
        seg_frames = [os.path.join(frames_folder, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
        first_name = os.path.basename(seg_frames[0])
        feat = get_segment_feature(video_category, video_name, first_name, seg_frames)
        xb = torch.tensor(feat, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            logit = float(model_obj(xb).squeeze().cpu().item())
        p = 1.0/(1.0+math.exp(-logit))
        probs.append((si, p))
    # group contiguous segments >= threshold into spans
    spans, span_scores = [], []
    cur = None
    cur_scores = []
    for si, p in probs:
        ssec, esec = seg_time_range(si)
        if p >= threshold:
            if cur is None:
                cur = [ssec, esec]; cur_scores = [p]
            else:
                cur[1] = esec; cur_scores.append(p)
        else:
            if cur is not None:
                spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
                cur = None; cur_scores = []
    if cur is not None:
        spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
    return spans, span_scores

# -----------------------
# Heuristic scoring to pick best model for a video
# -----------------------
def score_spans(spans, scores):
    if not spans:
        return -1e9
    durations = [max(0.0, e-s) for (s,e) in spans]
    avg_conf = float(np.mean(scores)) if scores else 0.0
    tiny = sum(1 for d in durations if d < 3.0)
    num = len(spans)
    total_dur = sum(durations)
    return (avg_conf * 2.0) + (total_dur * 0.02) - (tiny * 0.6) - (num * 0.05)

# -----------------------
# Build readable paragraph for video (non-LLM)
# - Prefer to use ground-truth sentences from UCF if they overlap with a span
# - Otherwise use short factual templates with timestamps
# -----------------------
def build_paragraph_nonllm(video_name, category, spans):
    if not spans:
        return f"No clear suspicious activity detected in {video_name}."
    gt = ucf_all.get(video_name, {})
    gt_times = gt.get("timestamps", []) if isinstance(gt, dict) else []
    gt_sents = gt.get("sentences", []) if isinstance(gt, dict) else []
    lines = []
    for (s, e) in spans:
        ts = sec_to_mmss(s)
        # find overlapping GT sentences
        matched = []
        for i, pair in enumerate(gt_times):
            try:
                gs, ge = float(pair[0]), float(pair[1])
            except Exception:
                continue
            if not (ge < s or gs > e):
                if i < len(gt_sents):
                    matched.append(gt_sents[i].strip().rstrip("."))
        if matched:
            text = " ".join(matched)
            lines.append(f"At around {ts}, {text}.")
        else:
            lines.append(f"At around {ts}, a suspicious {category.lower()} activity is detected between {sec_to_mmss(s)} and {sec_to_mmss(e)}.")
    paragraph = " ".join(lines)
    paragraph = re.sub(r"\s{2,}", " ", paragraph).strip()
    return paragraph

# -----------------------
# OPTIONAL: LLM polishing (commented out by default)
# If you have a compatible transformers install and want to enable LLM polishing,
# set USE_LLM = True. Note: may require installing specific transformer version.
# -----------------------
USE_LLM = False
llm_tokenizer = None
llm_model = None
def try_init_llm():
    global llm_tokenizer, llm_model
    if not USE_LLM:
        return False
    try:
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        MODEL_NAME = "google/flan-t5-base"  # lighter, should work on Kaggle if transformers compatible
        llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        llm_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
        return True
    except Exception as e:
        print("LLM init failed:", e)
        return False

def polish_with_llm(paragraph):
    if not USE_LLM:
        return paragraph
    if llm_model is None:
        if not try_init_llm():
            return paragraph
    prompt = f"Polish this surveillance summary to be professional, concise and factual, keep timestamps and do not hallucinate:\n\n{paragraph}"
    toks = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = llm_model.generate(**toks, max_new_tokens=200, num_beams=4)
    res = llm_tokenizer.decode(out[0], skip_special_tokens=True)
    return res.strip()

# -----------------------
# Build list of all videos from BASE_FRAMES
# -----------------------
if not os.path.isdir(BASE_FRAMES):
    raise RuntimeError(f"Frames folder not found: {BASE_FRAMES}")

all_videos = []
for cat in sorted(os.listdir(BASE_FRAMES)):
    cat_dir = os.path.join(BASE_FRAMES, cat)
    if not os.path.isdir(cat_dir):
        continue
    for vid in sorted(os.listdir(cat_dir)):
        if os.path.isdir(os.path.join(cat_dir, vid)):
            all_videos.append((cat, vid))

print("Videos to process:", len(all_videos))

# -----------------------
# Main loop: for each video, run each loaded model, pick best, save JSON + paragraph
# -----------------------
for (cat, vid) in tqdm(all_videos, desc="Videos"):
    frames_folder = os.path.join(BASE_FRAMES, cat, vid)
    # run all loaded models
    candidates = []
    for (name, path, model_obj) in loaded_models:
        try:
            spans, scores = infer_spans_with_model(model_obj, cat, vid, frames_folder, threshold=PROB_THRESHOLD)
        except Exception as e:
            print(f"Model {name} failed on {vid}: {e}")
            spans, scores = [], []
        sc = score_spans(spans, scores)
        candidates.append((sc, name, path, spans, scores))
    # choose best candidate by heuristic
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_name, best_path, best_spans, best_scores = candidates[0]

    # Save events JSON
    out_events = {
        "video_name": vid,
        "category": cat,
        "model_used": best_name,
        "avg_span_confidence": float(np.mean(best_scores)) if best_scores else 0.0,
        "predicted_events": [{"start": float(s), "end": float(e)} for (s,e) in best_spans]
    }
    with open(os.path.join(OUT_EVENTS_DIR, f"{vid}.json"), "w") as fh:
        json.dump(out_events, fh, indent=2)

    # Build paragraph (non-LLM) and optionally polish with LLM
    paragraph = build_paragraph_nonllm(vid, cat, best_spans)
    paragraph = polish_with_llm(paragraph)  # no-op if USE_LLM=False

    # Save paragraph
    para_path = os.path.join(OUT_PARAS_DIR, f"{vid}_summary.txt")
    with open(para_path, "w") as fh:
        fh.write(paragraph + "\n")

    # Print quick summary
    print("\n---")
    print(f"{vid} [{cat}]  model:{best_name}  events:{len(best_spans)}  avg_conf:{out_events['avg_span_confidence']:.3f}")
    print(paragraph)

print("\n✅ All done.")
print("Event JSONs:", OUT_EVENTS_DIR)
print("Paragraphs :", OUT_PARAS_DIR)

# ***Final Good one Stage 3***

In [ ]:
# ============================================================
#                STAGE 3 — AGENTIC VIDEO SUMMARIZATION
#   Uses:
#      ✔ Stage-1 features  (/kaggle/input/theft-features)
#      ✔ Stage-2 predicted events (/kaggle/input/stage1and2/events_v2)
#      ✔ Stage-2 best model not required at inference (events already given)
#      ✔ UCFCrime captions used for grounding
#   Produces:
#      ✔ High-quality agentic summaries (Generate → Reflect → Refine)
# ============================================================

!pip -q install transformers==4.41.2 accelerate==0.32.1

import os, json, torch, numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------
# CONFIG
# -------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "google/flan-t5-large"       # stronger LLM; can use flan-t5-xl if RAM ok

EVENTS_DIR = "/kaggle/input/stage1and2/events_v2"
FEATURES_DIR = "/kaggle/input/theft-features"
UCF_BASE = "/kaggle/input/crime-data/theft"
OUT_DIR = "/kaggle/working/final_agentic_summaries"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------
# LOAD LLM
# -------------------------------
print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

def ask_llm(prompt, max_tokens=200):
    tok = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = model.generate(**tok, max_new_tokens=max_tokens, num_beams=6)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def sec_to_minsec(t):
    m = int(t // 60)
    s = int(t % 60)
    return f"{m:02d}:{s:02d}"

# -------------------------------
# LOAD ALL UCF SENTENCES (Train+Val+Test)
# -------------------------------
ucf_all = {}
for fn in ["UCFCrime_Train.json", "UCFCrime_Val.json", "UCFCrime_Test.json"]:
    p = os.path.join(UCF_BASE, fn)
    if os.path.exists(p):
        with open(p) as f:
            d = json.load(f)
            if isinstance(d, dict):
                ucf_all.update(d)

print("Total GT caption videos loaded:", len(ucf_all))

# -------------------------------
# MAIN SUMMARIZATION LOOP
# -------------------------------
event_files = sorted([f for f in os.listdir(EVENTS_DIR) if f.endswith(".json")])
print("Event files found:", len(event_files))

for jf in tqdm(event_files):
    path = os.path.join(EVENTS_DIR, jf)
    ev = json.load(open(path))

    video = ev.get("video_name")
    category = ev.get("category", "Unknown")
    events = ev.get("predicted_events", [])

    if not events:
        continue

    # --- Load Stage-1 Feature (Video-level context)
    feat_path = os.path.join(FEATURES_DIR, category, f"{video}.pt")
    visual_context = ""
    if os.path.exists(feat_path):
        try:
            feat = torch.load(feat_path)
            feat_np = feat.cpu().numpy()
            visual_context = f"Visual feature summary vector length: {len(feat_np)}."
        except:
            visual_context = "Visual features could not be loaded."
    else:
        visual_context = "No visual feature available."

    # --- Try to get ground-truth sentences
    gt = ucf_all.get(video, {})
    gt_times = gt.get("timestamps", [])
    gt_sents = gt.get("sentences", [])

    event_lines = []

    for e in events:
        s, e_sec = float(e["start"]), float(e["end"])
        ts = sec_to_minsec(s)

        # Matching GT sentences
        matched = []
        for (t, sent) in zip(gt_times, gt_sents):
            if not t or len(t) < 2: 
                continue
            gs, ge = t
            if not (ge < s or gs > e_sec):
                matched.append(sent.strip())

        gt_text = " ".join(matched) if matched else "No direct GT caption."

        # ====================================================
        #         AGENTIC LLM — 3-STEP REASONING
        # ====================================================

        # 1️⃣ GENERATE
        p1 = f"""
You are an expert surveillance-video summarizer.
Category: {category}
Event window: {ts} ({s:.1f}s to {e_sec:.1f}s)
Visual context: {visual_context}
Reference caption: {gt_text}

Describe what likely happens in ONE clear factual sentence.
"""
        gen = ask_llm(p1, 120)

        # 2️⃣ REFLECT / CRITIQUE
        p2 = f"""
Improve clarity and correctness.

Sentence: "{gen}"

Rewrite it to be more accurate, concise, and visually grounded.
"""
        refined = ask_llm(p2, 120)

        # 3️⃣ POLISH
        p3 = f"""
Polish this into a professional surveillance description.
Keep ONE sentence only.

Text: {refined}
"""
        final_line = ask_llm(p3, 120)

        event_lines.append(f"At around {ts}, {final_line}")

    # -------------------------------
    # MERGE PARAGRAPH
    # -------------------------------
    merged = " ".join(event_lines)

    final_prompt = f"""
Combine these event lines into a smooth, chronological, 5–7 line paragraph
describing the entire video professionally.

Video: {video}
Category: {category}

{merged}
"""
    full_para = ask_llm(final_prompt, 250)

    out_path = os.path.join(OUT_DIR, f"{video}_summary.txt")
    with open(out_path, "w") as f:
        f.write(full_para.strip())

print("✅ Agentic summaries saved to:", OUT_DIR)

In [ ]:
# Stage 3 — Best-of-both inference + agentic LLM summarization (single cell)
# Paste into a Kaggle notebook and run. Adjust paths at top if you moved files.

# Optional installs (uncomment if needed)
# !pip -q install transformers==4.41.2 accelerate==0.32.1 --quiet

import os, json, math, re, warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from transformers import VideoMAEImageProcessor, VideoMAEModel, AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings("ignore")

# -----------------------
# CONFIG (edit only if needed)
# -----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Where frames are (stage-0)
BASE_FRAMES = "/kaggle/input/theft-frames"   # Category/Video/frame_*.jpg

# Candidate seg-cache & stage-2 event caches (prefer these)
CAND_SEG_CACHE = [
    "/kaggle/input/stage1and2/seg_feat_cache_v2",
    "/kaggle/input/stage1and2/seg-feat-cache-v2",
    "/kaggle/input/seg-feat-cache-v2",
    "/kaggle/working/seg_feat_cache_v2",
    "/kaggle/working/seg_feat_cache"
]
SEG_CACHE_DIR = next((p for p in CAND_SEG_CACHE if os.path.isdir(p)), None)

# Candidate Stage-2 checkpoint paths (where you put them)
CAND_MODEL_PATHS = [
    "/kaggle/input/newdata/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/newdata/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_Transformer_v2.pt",
    "/kaggle/working/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_model_v2.pt"
]

# Stage-1 video-level features (fallback)
VIDEO_FEATURES_DIR = "/kaggle/input/theft-features"  # your VideoMAE per-video .pt features

# UCFCrime JSONs (to read GT timestamps & sentences)
UCF_JSON_CAND = [
    "/kaggle/input/crime-data/theft/UCFCrime_Val.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Test.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Train.json"
]

# Output (working)
OUT_EVENTS_DIR = "/kaggle/working/events_best"
OUT_PARAS_DIR  = "/kaggle/working/final_paragraphs_stage32"
os.makedirs(OUT_EVENTS_DIR, exist_ok=True)
os.makedirs(OUT_PARAS_DIR, exist_ok=True)

# Segmenting params (must match Stage-2)
SEG_LEN_FRAMES = 16
SEG_STRIDE = 8
SAMPLE_EVERY = 10.0  # how many frames were skipped during stage-0 extraction
EFF_FPS = 30.0 / SAMPLE_EVERY

# LLM to use for sentence generation & polishing
LLM_MODEL = "google/flan-t5-base"  # change to larger model if you have resources
MAX_NEW_TOKENS = 160

# -----------------------
# Utility functions
# -----------------------
def sec_to_mmss(s):
    s = max(0.0, float(s))
    m = int(s // 60)
    ss = int(round(s % 60))
    return f"{m:02d}:{ss:02d}"

def seg_time_range(seg_idx, seg_len=SEG_LEN_FRAMES, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    start_frame = seg_idx * stride
    end_frame = start_frame + seg_len - 1
    return start_frame / eff_fps, end_frame / eff_fps

def iou(a_s, a_e, b_s, b_e):
    inter_s = max(a_s, b_s)
    inter_e = min(a_e, b_e)
    inter = max(0.0, inter_e - inter_s)
    union = (a_e - a_s) + (b_e - b_s) - inter
    return inter / union if union > 0 else 0.0

# -----------------------
# Define Stage-2 model classes (must match saved checkpoint keys)
# -----------------------
class TransformerClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        z = self.proj(x).unsqueeze(1)   # (B,1,d)
        z = self.encoder(z)             # (B,1,d)
        z = z.mean(dim=1)               # (B,d)
        return self.head(z).squeeze(1)  # (B,)

class BiLSTMClassifierV2(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk = feat_dim // self.seq_len
        self.lstm = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden*2, 1)
    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        B, D = x.shape
        x = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.head(out).squeeze(1)

# -----------------------
# Try load Stage-2 checkpoints (returns list of (name, path, model))
# -----------------------
loaded_models = []
for p in CAND_MODEL_PATHS:
    if not os.path.exists(p):
        continue
    try:
        state = torch.load(p, map_location="cpu")
    except Exception as e:
        print("Failed to torch.load", p, ":", e)
        continue

    # try Transformer
    try:
        m = TransformerClassifierV2().to(DEVICE)
        try:
            m.load_state_dict(state, strict=True)
        except Exception:
            # try remap cls.* -> head.* keys (older/newer savings)
            remapped = {k.replace("cls.", "head."): v for k,v in state.items()}
            m.load_state_dict(remapped, strict=False)
        m.eval()
        loaded_models.append(("Transformer", p, m))
        print("Loaded Transformer from:", p)
        continue
    except Exception:
        pass

    # try BiLSTM
    try:
        m = BiLSTMClassifierV2().to(DEVICE)
        try:
            m.load_state_dict(state, strict=True)
        except Exception:
            remapped = {k.replace("cls.", "head."): v for k,v in state.items()}
            m.load_state_dict(remapped, strict=False)
        m.eval()
        loaded_models.append(("BiLSTM", p, m))
        print("Loaded BiLSTM from:", p)
        continue
    except Exception:
        pass

    print("Could not load model at", p)

if not loaded_models:
    raise RuntimeError("No Stage-2 checkpoints found in candidate paths. Put checkpoints in /kaggle/input or /kaggle/working.")

print("Stage-2 models available:", [m[0] for m in loaded_models])

# -----------------------
# Segment feature loader: prefer SEG_CACHE_DIR -> VIDEO_FEATURES_DIR -> compute (VideoMAE)
# -----------------------
processor = None
vmodel = None
def ensure_videomae():
    global processor, vmodel
    if processor is None or vmodel is None:
        processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
        vmodel = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def seg_cache_path(video_key, first_frame_name):
    if not SEG_CACHE_DIR:
        return None
    candidate = f"{video_key}__{first_frame_name}".replace(".", "_") + ".npy"
    return os.path.join(SEG_CACHE_DIR, candidate)

def load_segment_feature_from_cache(video_key, first_frame_name):
    p = seg_cache_path(video_key, first_frame_name)
    if p and os.path.exists(p):
        try:
            arr = np.load(p, allow_pickle=False)
            arr = np.array(arr, dtype=np.float32).reshape(-1)
            return arr
        except Exception:
            return None
    return None

def load_video_level_feature(video_category, video_name):
    if not os.path.isdir(VIDEO_FEATURES_DIR):
        return None
    cand = os.path.join(VIDEO_FEATURES_DIR, video_category, f"{video_name}.pt")
    if not os.path.exists(cand):
        cand2 = os.path.join(VIDEO_FEATURES_DIR, f"{video_name}.pt")
        cand = cand2 if os.path.exists(cand2) else cand
    if os.path.exists(cand):
        try:
            feat = torch.load(cand, map_location="cpu")
            if isinstance(feat, torch.Tensor):
                feat = feat.mean(dim=0).cpu().numpy() if feat.ndim>1 else feat.cpu().numpy()
            feat = np.array(feat, dtype=np.float32).reshape(-1)
            return feat
        except Exception:
            return None
    return None

def compute_segment_feature_from_frames(frames_paths):
    ensure_videomae()
    imgs = []
    for p in frames_paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not imgs:
        return np.zeros((768,), dtype=np.float32)
    with torch.no_grad():
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out = vmodel(**inputs).last_hidden_state
        feat = out.mean(dim=1).mean(dim=0).float().cpu().numpy()
    return np.array(feat, dtype=np.float32).reshape(-1)

def get_segment_feature(video_category, video_name, first_frame_name, frames_paths):
    # 1) seg cache
    f = load_segment_feature_from_cache(video_name, first_frame_name)
    if f is not None:
        return f
    # 2) video-level features (replicate)
    fv = load_video_level_feature(video_category, video_name)
    if fv is not None:
        return fv
    # 3) compute (slow)
    return compute_segment_feature_from_frames(frames_paths)

# -----------------------
# Load UCFCrime JSONs (ground truth sentences/timestamps)
# -----------------------
ucf_all = {}
for jp in UCF_JSON_CAND:
    try:
        if os.path.exists(jp):
            d = json.load(open(jp))
            if isinstance(d, dict):
                ucf_all.update(d)
            else:
                # if a list, try to index by filename keys if present
                for item in d:
                    key = item.get("video_name") or item.get("id") or item.get("name")
                    if key:
                        ucf_all[key] = item
            print("Loaded UCF JSON:", jp, "-> total entries:", len(ucf_all))
    except Exception:
        pass

# -----------------------
# Inference function per model (returns spans and avg scores)
# -----------------------
def infer_spans_with_model(model, video_category, video_name, frames_folder, threshold=0.5):
    frames = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if len(frames) < SEG_LEN_FRAMES:
        return [], []
    n_segs = max(1, (len(frames) - SEG_LEN_FRAMES)//SEG_STRIDE + 1)
    probs = []
    for si in range(n_segs):
        start_idx = si * SEG_STRIDE
        seg_frames = [os.path.join(frames_folder, frames[i]) for i in range(start_idx, min(start_idx+SEG_LEN_FRAMES, len(frames)))]
        first_name = os.path.basename(seg_frames[0])
        feat = get_segment_feature(video_category, video_name, first_name, seg_frames)
        xb = torch.tensor(feat, dtype=torch.float32, device=DEVICE).unsqueeze(0)  # (1,768)
        with torch.no_grad():
            logit = float(model(xb).squeeze().cpu().item())
        p = 1.0/(1.0+math.exp(-logit))
        probs.append((si, p))
    # group contiguous positives
    spans, span_scores = [], []
    cur = None; cur_scores = []
    for si, p in probs:
        ssec, esec = seg_time_range(si)
        if p >= threshold:
            if cur is None:
                cur = [ssec, esec]; cur_scores = [p]
            else:
                cur[1] = esec; cur_scores.append(p)
        else:
            if cur is not None:
                spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
                cur = None; cur_scores = []
    if cur is not None:
        spans.append(tuple(cur)); span_scores.append(float(np.mean(cur_scores)))
    return spans, span_scores

# -----------------------
# Score spans to pick best model per video
# -----------------------
def score_spans(spans, scores):
    if not spans:
        return -1e9
    durations = [max(0.0, e-s) for (s,e) in spans]
    avg_conf = float(np.mean(scores)) if scores else 0.0
    tiny = sum(1 for d in durations if d < 3.0)
    num = len(spans)
    total_dur = sum(durations)
    return (avg_conf * 2.0) + (total_dur * 0.02) - (tiny * 0.6) - (num * 0.05)

# -----------------------
# Load LLM (Flan-T5) for sentence generation & polishing
# -----------------------
print("Loading LLM:", LLM_MODEL)
tokenizer_llm = AutoTokenizer.from_pretrained(LLM_MODEL)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL).to(DEVICE).eval()

def ask_llm(prompt, max_new_tokens=MAX_NEW_TOKENS):
    toks = tokenizer_llm(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = llm_model.generate(**toks, max_new_tokens=max_new_tokens, num_beams=3)
    return tokenizer_llm.decode(out[0], skip_special_tokens=True).strip()

# -----------------------
# Build one-sentence description for each span.
# Use GT sentences if available; otherwise ask LLM with template + optional sample frames names.
# -----------------------
def describe_span(video_name, category, s, e, frames_folder, sample_frame_paths=None):
    # try ground truth sentences
    gt = ucf_all.get(video_name, {})
    gt_times = gt.get("timestamps", []) if isinstance(gt, dict) else gt.get("timestamps", [])
    gt_sents = gt.get("sentences", []) if isinstance(gt, dict) else gt.get("sentences", [])
    matched = []
    for i, ts in enumerate(gt_times):
        if not ts or len(ts) < 2:
            continue
        gs, ge = float(ts[0]), float(ts[1])
        if not (ge < s or gs > e):
            if i < len(gt_sents):
                matched.append(gt_sents[i].strip().rstrip("."))
    if matched:
        return " ".join(matched).strip() + "."
    # else use LLM
    time_label = sec_to_mmss(s)
    visual_hint = ""
    if sample_frame_paths:
        # include small hint of filenames (LLM won't see images but gets context)
        sample_names = [os.path.basename(p) for p in sample_frame_paths[:3]]
        visual_hint = " Visual examples: " + ", ".join(sample_names) + "."
    prompt = (
        f"You are a concise, factual surveillance summarizer. "
        f"Video: {video_name} (category: {category}). "
        f"Time window approx {time_label} ({s:.1f}s–{e:.1f}s). "
        f"Describe in one short factual sentence what likely happens in that clip.{visual_hint} "
        f"Keep the sentence objective and avoid speculation."
    )
    out = ask_llm(prompt, max_new_tokens=70)
    # ensure punctuation
    out = out.strip()
    if not out.endswith("."):
        out = out + "."
    return out

# -----------------------
# Build full paragraph for a video (LLM polishes chronological sentences)
# -----------------------
def build_paragraph(video_name, category, spans, frames_folder):
    if not spans:
        return f"No clear suspicious activity detected in {video_name}."
    # create one-lines
    lines = []
    for (s,e) in spans:
        # choose sample frame paths if available
        sample_frames = None
        try:
            frames = sorted([f for f in os.listdir(frames_folder) if f.lower().endswith(('.jpg','.jpeg','.png'))])
            start_idx = max(0, int(round(s * EFF_FPS)))
            sample_frames = [os.path.join(frames_folder, frames[min(start_idx + i, len(frames)-1)]) for i in range(0, min(4, len(frames)) , max(1, SEG_LEN_FRAMES//4))]
        except Exception:
            sample_frames = None
        sent = describe_span(video_name, category, s, e, frames_folder, sample_frames)
        lines.append(f"At around {sec_to_mmss(s)}, {sent}")
    # ask LLM to merge into polished paragraph
    combined = " ".join(lines)
    prompt = (
        "Combine the following time-tagged sentences into a short, fluent, and professional paragraph (2-6 sentences). "
        "Keep timestamps as 'At around MM:SS'. Be factual and concise.\n\n"
        f"{combined}"
    )
    final = ask_llm(prompt, max_new_tokens=160)
    return final

# -----------------------
# Iterate all videos, run models, pick best model per video, save outputs
# -----------------------
# Gather list of videos (Category, VideoName pairs)
all_videos = []
if not os.path.isdir(BASE_FRAMES):
    raise RuntimeError(f"BASE_FRAMES not found: {BASE_FRAMES}")
for cat in sorted(os.listdir(BASE_FRAMES)):
    cat_dir = os.path.join(BASE_FRAMES, cat)
    if not os.path.isdir(cat_dir):
        continue
    for vid in sorted(os.listdir(cat_dir)):
        vdir = os.path.join(cat_dir, vid)
        if os.path.isdir(vdir):
            all_videos.append((cat, vid))
print("Videos to process:", len(all_videos))

for (cat, vid) in tqdm(all_videos, desc="Videos"):
    frames_folder = os.path.join(BASE_FRAMES, cat, vid)

    # run every loaded model and collect spans
    candidates = []
    for (name, path, model) in loaded_models:
        spans, scores = infer_spans_with_model(model, cat, vid, frames_folder, threshold=0.5)
        sc = score_spans(spans, scores)
        candidates.append((sc, name, path, spans, scores))
    # pick best candidate (highest score)
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_score, best_name, best_path, best_spans, best_scores = candidates[0]

    # build paragraph (LLM polishing)
    paragraph = build_paragraph(vid, cat, best_spans, frames_folder)

    # save events JSON and paragraph
    out_events = {
        "video_name": vid,
        "category": cat,
        "model_used": best_name,
        "avg_span_confidence": float(np.mean(best_scores)) if best_scores else 0.0,
        "predicted_events": [{"start": float(s), "end": float(e)} for (s,e) in best_spans]
    }
    with open(os.path.join(OUT_EVENTS_DIR, f"{vid}.json"), "w") as fh:
        json.dump(out_events, fh, indent=2)

    out_txt_path = os.path.join(OUT_PARAS_DIR, f"{vid}_summary.txt")
    with open(out_txt_path, "w") as fh:
        fh.write(paragraph.strip() + "\n")

    # quick console print for verification
    print("\n==============================")
    print(f"🎬 {vid} [{cat}] | Model: {best_name} | Events: {len(best_spans)} | Score: {best_score:.3f}")
    print(paragraph)

print("\n✅ Done.")
print("Event JSONs:", OUT_EVENTS_DIR)
print("Paragraphs :", OUT_PARAS_DIR)

In [ ]:
import shutil

# Path where your processed frames were saved
final_dir = "/kaggle/working/"
zip_path = "/kaggle/working/stage3.zip"

# Create zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', final_dir)
print("✅ Zipped successfully!")

# Download (for Kaggle)
from IPython.display import FileLink
FileLink(zip_path)

# ***Stage 4 - Full Pipeline: From Video → Paragraph Summary***

In [ ]:
# ===============================================================
# Stage 4 — End-to-End Agentic LLM Summarization for New Video
# ===============================================================

!pip install -q transformers accelerate timm

import os, cv2, shutil, torch, numpy as np
from tqdm import tqdm
from PIL import Image
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    VideoMAEImageProcessor, VideoMAEModel,
    BlipProcessor, BlipForConditionalGeneration
)

# -------------------------------
# CONFIG
# -------------------------------
VIDEO_PATH = "/kaggle/input/crime-data/theft/Robbery/Robbery005_x264.mp4"   # change for new input
TEMP_DIR = "/kaggle/working/temp_newvideo_frames"
MODEL_PATH_STAGE2 = "/kaggle/input/models/pytorch/default/1/best_event_classifier_vit.pt"
FEATURES_DIR = "/kaggle/input/theft-features"
OUT_FILE = "/kaggle/working/final_summary_output.txt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAME_INTERVAL = 10
FRAME_SIZE = (224, 224)
LLM_NAME = "google/flan-t5-large"

print(f"🔹 Using device: {DEVICE}")

# -------------------------------
# Step 0: Extract frames fresh
# -------------------------------
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)
os.makedirs(TEMP_DIR, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
frames, frame_id, saved = [], 0, 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_id % FRAME_INTERVAL == 0:
        frame = cv2.resize(frame, FRAME_SIZE)
        frame_path = os.path.join(TEMP_DIR, f"frame_{saved:04d}.jpg")
        cv2.imwrite(frame_path, frame)
        frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        saved += 1
    frame_id += 1
cap.release()
print(f"✅ Extracted {saved} frames")

# -------------------------------
# Step 1: VideoMAE features
# -------------------------------
processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

inputs = processor(frames[:16], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = videomae(**inputs)
    video_feat = outputs.last_hidden_state.mean(dim=1).cpu().numpy().flatten()
print("✅ Feature dimension:", video_feat.shape[0])

# -------------------------------
# Step 2: Load Stage2 classifier
# -------------------------------
class EventClassifier(torch.nn.Module):
    def __init__(self, in_dim=768, hidden=512, num_classes=4):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden, num_classes)
        )
    def forward(self, x):
        if x.dim() == 1: x = x.unsqueeze(0)
        return self.net(x)

categories = ["Burglary", "Robbery", "Shoplifting", "Stealing"]
clf = EventClassifier().to(DEVICE)
clf.load_state_dict(torch.load(MODEL_PATH_STAGE2, map_location=DEVICE))
clf.eval()

x = torch.tensor(video_feat).float().to(DEVICE)
with torch.no_grad():
    pred = clf(x).argmax().item()
event_label = categories[pred]
print("🧠 Initial predicted category:", event_label)

# -------------------------------
# Step 3: Verify by cosine similarity
# -------------------------------
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

means = {}
for cat in categories:
    cat_dir = os.path.join(FEATURES_DIR, cat)
    feats = []
    for f in os.listdir(cat_dir):
        if f.endswith(".pt"):
            arr = torch.load(os.path.join(cat_dir, f))
            if isinstance(arr, torch.Tensor): arr = arr.cpu().numpy()
            if arr.ndim > 1: arr = arr.mean(axis=0)
            feats.append(arr)
    if feats:
        means[cat] = np.mean(feats, axis=0)

sims = {c: cosine_sim(video_feat, v) for c, v in means.items()}
best_cat = max(sims, key=sims.get)
if best_cat != event_label and sims[best_cat] > sims[event_label] + 0.05:
    print(f"🧭 Corrected category: {best_cat} (was {event_label})")
    event_label = best_cat
else:
    print("✅ Category confirmed:", event_label)

# -------------------------------
# Step 4: Add BLIP visual context
# -------------------------------
blip_proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(DEVICE)

def describe_frame(img):
    inputs = blip_proc(img, return_tensors="pt").to(DEVICE)
    out = blip_model.generate(**inputs, max_new_tokens=40)
    return blip_proc.decode(out[0], skip_special_tokens=True)

sample_idxs = [0, len(frames)//2, -1]
sample_captions = [describe_frame(frames[i]) for i in sample_idxs]
context_text = " ".join(sample_captions)
print("🖼️ Visual context:", context_text)

# -------------------------------
# Step 5: Agentic LLM Summarization
# -------------------------------
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(DEVICE).eval()

def ask_llm(prompt, max_tokens=128):
    t = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    with torch.no_grad():
        out = llm.generate(**t, max_new_tokens=max_tokens, num_beams=6, do_sample=True, temperature=0.9)
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def sec_to_mmss(sec): 
    return f"{int(sec//60):02d}:{int(sec%60):02d}"

duration = (saved * FRAME_INTERVAL) / fps
segments = np.linspace(0, duration, 6)
events = [{"start": float(segments[i]), "end": float(segments[i+1])}
          for i in range(len(segments)-1)]

lines = []
for i, ev in enumerate(events):
    s, e = ev["start"], ev["end"]
    t_str = sec_to_mmss(s)
    pos = "beginning" if i == 0 else "middle" if i < len(events)//2 else "end"

    prompt = (
        f"Context: {context_text}\n"
        f"You are analyzing a {event_label.lower()} surveillance video. "
        f"This is the {pos} part of the clip ({s:.1f}s–{e:.1f}s). "
        f"Describe briefly what is likely happening, in one realistic, unique sentence."
    )
    desc = ask_llm(prompt, max_tokens=80)

    refine_prompt = (
        f"Polish the following event description for clarity and fluency:\n"
        f"'{desc}'\n\nReturn one natural, factual sentence."
    )
    refined = ask_llm(refine_prompt, max_tokens=60)
    lines.append(f"At around {t_str}, {refined}")

final_prompt = (
    f"Combine these time-stamped observations into a coherent paragraph summarizing "
    f"this {event_label.lower()} video. Merge related sentences, keep chronological order, "
    f"and make it flow naturally.\n\n{' '.join(lines)}\n\n"
    "Return a single short paragraph (4–6 sentences)."
)

final_summary = ask_llm(final_prompt, max_tokens=220)

# -------------------------------
# Step 6: Save and show
# -------------------------------
print("\n🧩 Final Summary:\n")
print(final_summary)

with open(OUT_FILE, "w") as f:
    f.write(final_summary)

print(f"\n✅ Summary saved to: {OUT_FILE}")

# ***Stage 4 New***

In [ ]:
# Single-cell: STAGE 4 — FINAL VIDEO SUMMARIZER (upload video -> summary)
# Paste into Kaggle notebook and run. Edit VIDEO_PATH below to test with one video.

!pip -q install transformers==4.41.2 accelerate==0.32.1 opencv-python --quiet

import os, math, json, tempfile, warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image
import cv2
import torch
import torch.nn as nn
from transformers import (
    VideoMAEImageProcessor, VideoMAEModel,
    AutoTokenizer, AutoModelForSeq2SeqLM
)

warnings.filterwarnings("ignore")

# -------------------------
# CONFIG (edit only these)
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VIDEO_PATH = "/kaggle/input/crime-data/theft/Robbery/Robbery007_x264.mp4"  # <-- change to test video
OUT_DIR = "/kaggle/working/final_stage4_output"
os.makedirs(OUT_DIR, exist_ok=True)

# Candidate model checkpoint locations (your provided paths)
CAND_TRANS = [
    "/kaggle/input/newdata/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_Transformer_v2.pt",
    "/kaggle/working/best_seg_Transformer_v2.pt",
    "/kaggle/working/best_seg_model_v2.pt"
]
CAND_BILSTM = [
    "/kaggle/input/newdata/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/input/models/pytorch/default/1/best_seg_BiLSTM_v2.pt",
    "/kaggle/working/best_seg_BiLSTM_v2.pt"
]

# Candidate seg-feature cache dirs / stage1 features / stage2 events (from your environment)
SEG_CACHE_CAND = [
    "/kaggle/input/stage1and2/seg_feat_cache_v2",
    "/kaggle/input/stage1and2/seg-feat-cache-v2",
    "/kaggle/input/seg-feat-cache-v2",
    "/kaggle/working/seg_feat_cache_v2",
]
VIDEO_FEATURES_DIR = "/kaggle/input/theft-features"   # stage1 per-video features (optional)
UCF_JSONS = [
    "/kaggle/input/crime-data/theft/UCFCrime_Test.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Val.json",
    "/kaggle/input/crime-data/theft/UCFCrime_Train.json"
]

# Segmentation params (must match Stage-2)
SAMPLE_EVERY = 10.0   # Stage-0 sampled every 10 frames
EFF_FPS = 30.0 / SAMPLE_EVERY   # effective fps mapping frames -> seconds
SEG_LEN = 16
SEG_STRIDE = 8

# LLM config (smaller for Kaggle; swap if you have more memory)
LLM_NAME = "google/flan-t5-base"
LLM_BEAMS = 3
LLM_MAX_TOKENS_SENT = 80
LLM_MAX_TOKENS_PAR = 200

print("DEVICE:", DEVICE)

# -------------------------
# Helpers
# -------------------------
def sec_to_mmss(s):
    s = max(0.0, float(s))
    m = int(s // 60); ss = int(round(s % 60))
    return f"{m:02d}:{ss:02d}"

def seg_time_range(seg_idx, seg_len=SEG_LEN, stride=SEG_STRIDE, eff_fps=EFF_FPS):
    start_frame = seg_idx * stride
    end_frame = start_frame + seg_len - 1
    return start_frame / eff_fps, end_frame / eff_fps

def sigmoid(x): return 1.0 / (1.0 + math.exp(-x))

# -------------------------
# Stage-2 model classes (match saved checkpoints shape variations)
# -------------------------
class TransformerClassifier(nn.Module):
    def __init__(self, feat_dim=768, d_model=256, nhead=4, num_layers=2):
        super().__init__()
        self.proj = nn.Linear(feat_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        # use Sequential head to match saved variants (LayerNorm + Linear)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        if x.dim() == 1: x = x.unsqueeze(0)
        z = self.proj(x).unsqueeze(1)   # (B,1,d)
        z = self.enc(z)                 # (B,1,d)
        z = z.mean(dim=1)               # (B,d)
        out = self.head(z)              # (B,1)
        return out.squeeze(1)           # (B,)

class BiLSTMClassifier(nn.Module):
    def __init__(self, feat_dim=768, hidden=256):
        super().__init__()
        chunk = 64
        self.seq_len = max(1, feat_dim // chunk)
        self.chunk = feat_dim // self.seq_len
        self.lstm = nn.LSTM(self.chunk, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(hidden*2, 1)
    def forward(self, x):
        if x.dim() == 1: x = x.unsqueeze(0)
        B, D = x.shape
        x = x[:, :self.chunk*self.seq_len].reshape(B, self.seq_len, self.chunk)
        out, _ = self.lstm(x)
        out = out.mean(dim=1)
        return self.head(out).squeeze(1)

# -------------------------
# Robust checkpoint loader
# -------------------------
def try_load_checkpoint(candidate_paths, cls):
    for p in candidate_paths:
        if not os.path.exists(p): 
            continue
        try:
            sd = torch.load(p, map_location="cpu")
        except Exception as e:
            print("Torch.load failed:", p, "->", e); continue
        model = cls().to(DEVICE)
        # try strict True then False; try remapping cls. -> head. keys
        success = False
        try:
            model.load_state_dict(sd, strict=True)
            success = True; mode = "strict"
        except Exception:
            try:
                # remap any "cls." -> "head." or vice versa
                remap = {k.replace("cls.", "head."): v for k,v in sd.items()}
                model.load_state_dict(remap, strict=False)
                success = True; mode = "remapped"
            except Exception:
                try:
                    model.load_state_dict(sd, strict=False)
                    success = True; mode = "loose"
                except Exception as e:
                    success = False
        if success:
            print(f"Loaded {cls.__name__} from {p}  (mode: {mode})")
            return model, p
    return None, None

# -------------------------
# locate seg-cache & stage1 features
# -------------------------
SEG_CACHE_DIR = next((d for d in SEG_CACHE_CAND if os.path.isdir(d)), None)
print("SEG_CACHE_DIR:", SEG_CACHE_DIR)
print("VIDEO_FEATURES_DIR (stage1):", VIDEO_FEATURES_DIR)

# -------------------------
# Load Stage-2 models (robust)
# -------------------------
trans_model, trans_path = try_load_checkpoint(CAND_TRANS, TransformerClassifier)
bilstm_model, bilstm_path = try_load_checkpoint(CAND_BILSTM, BiLSTMClassifier)

if trans_model is None and bilstm_model is None:
    raise RuntimeError("No stage-2 checkpoints found. Place .pt in candidate paths.")

loaded_models = []
if trans_model is not None: loaded_models.append(("Transformer", trans_path, trans_model))
if bilstm_model is not None: loaded_models.append(("BiLSTM", bilstm_path, bilstm_model))
print("Stage-2 models available:", [m[0] for m in loaded_models])

# -------------------------
# VideoMAE (only if needed for computing features)
# -------------------------
processor = None
vmodel = None
def ensure_videomae():
    global processor, vmodel
    if processor is None or vmodel is None:
        processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")
        vmodel = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base").to(DEVICE).eval()

def load_segment_feature_from_cache(video_name, first_frame_name):
    if not SEG_CACHE_DIR:
        return None
    key = (video_name + "__" + first_frame_name).replace(".", "_") + ".npy"
    p = os.path.join(SEG_CACHE_DIR, key)
    if os.path.exists(p):
        try:
            arr = np.load(p, allow_pickle=False)
            return np.array(arr, dtype=np.float32).reshape(-1)
        except Exception:
            return None
    return None

def load_video_level_feature(video_category, video_name):
    # expect .pt saved per-video under VIDEO_FEATURES_DIR/category/video.pt or VIDEO_FEATURES_DIR/video.pt
    if not os.path.isdir(VIDEO_FEATURES_DIR):
        return None
    cand = os.path.join(VIDEO_FEATURES_DIR, video_category, f"{video_name}.pt")
    if not os.path.exists(cand):
        cand2 = os.path.join(VIDEO_FEATURES_DIR, f"{video_name}.pt")
        cand = cand2 if os.path.exists(cand2) else cand
    if os.path.exists(cand):
        try:
            feat = torch.load(cand, map_location="cpu")
            if isinstance(feat, torch.Tensor):
                feat = feat.mean(dim=0).cpu().numpy() if feat.ndim>1 else feat.cpu().numpy()
            return np.array(feat, dtype=np.float32).reshape(-1)
        except Exception:
            return None
    return None

def compute_segment_feature_from_frames(frame_paths):
    ensure_videomae()
    imgs = []
    for p in frame_paths:
        try:
            imgs.append(Image.open(p).convert("RGB"))
        except Exception:
            pass
    if not imgs:
        return np.zeros((768,), dtype=np.float32)
    with torch.no_grad():
        inputs = processor(images=imgs, return_tensors="pt").to(DEVICE)
        out = vmodel(**inputs).last_hidden_state
        feat = out.mean(dim=1).mean(dim=0).float().cpu().numpy()
    return np.array(feat, dtype=np.float32).reshape(-1)

def get_segment_feature(video_category, video_name, first_frame_name, frame_paths):
    # 1. segment cache
    f = load_segment_feature_from_cache(video_name, first_frame_name)
    if f is not None: return f
    # 2. per-video features
    fv = load_video_level_feature(video_category, video_name)
    if fv is not None: return fv
    # 3. compute (fallback)
    return compute_segment_feature_from_frames(frame_paths)

# -------------------------
# Load UCF captions (if present)
# -------------------------
ucf_all = {}
for p in UCF_JSONS:
    if os.path.exists(p):
        try:
            j = json.load(open(p))
            if isinstance(j, dict):
                ucf_all.update(j)
            else:
                for item in j:
                    key = item.get("video_name") or item.get("id") or item.get("name")
                    if key:
                        ucf_all[key] = item
            print("Loaded UCF JSON:", p, "entries:", len(ucf_all))
        except Exception:
            pass

# -------------------------
# LLM setup
# -------------------------
print("Loading LLM:", LLM_NAME)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to(DEVICE).eval()

def ask_llm(prompt, max_new_tokens=LLM_MAX_TOKENS_SENT):
    toks = tok(prompt, return_tensors="pt", truncation=True, max_length=1024).to(DEVICE)
    out = llm.generate(**toks, max_new_tokens=max_new_tokens, num_beams=LLM_BEAMS)
    return tok.decode(out[0], skip_special_tokens=True).strip()

# -------------------------
# Video -> frames extraction (samples every 10th frame to match training)
# -------------------------
TMP_FRAMES = tempfile.mkdtemp(prefix="stage4_frames_")
print("Extracting frames from", VIDEO_PATH)
cap = cv2.VideoCapture(VIDEO_PATH)
frame_idx = 0
saved = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    if frame_idx % int(SAMPLE_EVERY) == 0:
        fp = os.path.join(TMP_FRAMES, f"frame_{saved:05d}.jpg")
        cv2.imwrite(fp, frame)
        saved += 1
    frame_idx += 1
cap.release()
print("Frames saved:", saved)
if saved == 0:
    raise RuntimeError("No frames extracted from video.")

frames_list = sorted([os.path.join(TMP_FRAMES, f) for f in os.listdir(TMP_FRAMES) if f.lower().endswith(".jpg")])
N_frames = len(frames_list)
n_segs = max(1, (N_frames - SEG_LEN)//SEG_STRIDE + 1)
print("Segments to evaluate:", n_segs)

# -------------------------
# Inference: get spans & scores per model
# -------------------------
def infer_spans_for_model(model_tuple):
    name, path, model = model_tuple
    spans = []
    span_scores = []
    for si in range(n_segs):
        start_idx = si * SEG_STRIDE
        seg_frame_paths = frames_list[start_idx: start_idx + SEG_LEN]
        first_frame_name = os.path.basename(seg_frame_paths[0])
        feat = get_segment_feature(Path(VIDEO_PATH).parent.name, Path(VIDEO_PATH).stem, first_frame_name, seg_frame_paths)
        xb = torch.tensor(feat, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            logit = model(xb)
            # logit may be shape (1,) or (1,1)
            try:
                val = float(logit.squeeze().cpu().item())
            except Exception:
                val = float(logit.cpu().numpy().squeeze().item())
        p = sigmoid(val)
        if p >= 0.5:
            s, e = seg_time_range(si)
            spans.append((s, e))
            span_scores.append(p)
    # (also compute avg confidence per span group if needed)
    return spans, span_scores

model_spans = {}
for mt in loaded_models:
    spans, scores = infer_spans_for_model(mt)
    model_spans[mt[0]] = {"spans": spans, "scores": scores}
    print(f"Model {mt[0]} detected {len(spans)} segments.")

# -------------------------
# Score heuristic and pick best model
# -------------------------
def score_spans(spans, scores):
    if not spans: return -1e9
    durations = [max(0.0, e-s) for (s,e) in spans]
    avg_conf = float(np.mean(scores)) if scores else 0.0
    tiny = sum(1 for d in durations if d < 3.0)
    num = len(spans)
    total = sum(durations)
    return (avg_conf * 2.0) + (total * 0.02) - (tiny * 0.6) - (num * 0.05)

scores = {}
for name, info in model_spans.items():
    scores[name] = score_spans(info["spans"], info["scores"])
print("Model span scores:", scores)
chosen_model_name = max(scores, key=scores.get)
chosen_info = model_spans[chosen_model_name]
chosen_spans = chosen_info["spans"]
print("Chosen model:", chosen_model_name, "| spans:", chosen_spans)

# if no spans, fallback to whole video
if not chosen_spans:
    total_dur = n_segs * SEG_STRIDE / EFF_FPS
    chosen_spans = [(0.0, total_dur)]

# -------------------------
# Describe spans -> sentences (use GT captions if available, else LLM)
# -------------------------
video_basename = Path(VIDEO_PATH).stem
ucf_entry = ucf_all.get(video_basename, {})

def find_gt_sentences_for_span(s, e):
    if not ucf_entry:
        return []
    gts = ucf_entry.get("timestamps", [])
    sents = ucf_entry.get("sentences", [])
    matched = []
    for i, ts in enumerate(gts):
        if not ts or len(ts) < 2: continue
        gs, ge = float(ts[0]), float(ts[1])
        if not (ge < s or gs > e):
            if i < len(sents):
                matched.append(sents[i].strip().rstrip("."))
    return matched

lines = []
for (s,e) in chosen_spans:
    # sample a few filenames as visual hint (LLM can't see images)
    start_frame_idx = min(int(round(s * EFF_FPS)), N_frames-1)
    sample_idxs = [min(start_frame_idx + k, N_frames-1) for k in range(0, min(SEG_LEN, N_frames-start_frame_idx), max(1, SEG_LEN//4))]
    sample_files = [os.path.basename(frames_list[i]) for i in sample_idxs]
    visual_hint = (" Visual file samples: " + ", ".join(sample_files) + ".") if sample_files else ""

    gt_matched = find_gt_sentences_for_span(s, e)
    if gt_matched:
        sent = " ".join(gt_matched).strip()
        if not sent.endswith("."): sent = sent + "."
    else:
        # agentic LLM: generate -> reflect -> polish
        tlabel = sec_to_mmss(s)
        p1 = f"You are an objective surveillance-video summarizer.\nVideo: {video_basename}\nTime window approx: {tlabel} ({s:.1f}s–{e:.1f}s).{visual_hint}\nWrite ONE short factual sentence describing what happens in this clip. Be objective."
        gen = ask_llm(p1, max_new_tokens=LLM_MAX_TOKENS_SENT)
        p2 = f"Rewrite the sentence to be more concise and visually grounded (ONE sentence):\n{gen}"
        ref = ask_llm(p2, max_new_tokens=LLM_MAX_TOKENS_SENT)
        p3 = f"Polish this to a clear single professional sentence (no extra commentary):\n{ref}"
        sent = ask_llm(p3, max_new_tokens=LLM_MAX_TOKENS_SENT)
        sent = sent.strip()
        if not sent.endswith("."): sent = sent + "."
    lines.append(f"At around {sec_to_mmss(s)}, {sent}")

# -------------------------
# Merge lines into final paragraph via LLM
# -------------------------
combined = " ".join(lines)
merge_prompt = (
    "Combine the following time-tagged sentences into a fluent, professional, chronological paragraph "
    "(2-6 sentences). Keep timestamps like 'At around MM:SS' and be factual.\n\n"
    f"{combined}"
)
final_paragraph = ask_llm(merge_prompt, max_new_tokens=LLM_MAX_TOKENS_PAR)

# -------------------------
# Print & save
# -------------------------
print("\n===== FINAL SUMMARY =====\n")
print(final_paragraph)
print("\n=========================\n")
meta = {"model_used": chosen_model_name, "spans": chosen_spans, "video": video_basename}
with open(os.path.join(OUT_DIR, f"{video_basename}_summary.txt"), "w") as f:
    f.write(final_paragraph + "\n")
with open(os.path.join(OUT_DIR, f"{video_basename}_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print("Saved outputs to:", OUT_DIR)

In [ ]:
import shutil

# Path where your processed frames were saved
final_dir = "/kaggle/working/"
zip_path = "/kaggle/working/stage4.zip"

# Create zip file
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', final_dir)
print("✅ Zipped successfully!")

# Download (for Kaggle)
from IPython.display import FileLink
FileLink(zip_path)

# ***VideoMAE Evaluation of Results***

In [ ]:
# ==========================================
# VideoMAE FINAL EVALUATION (NO TRAINING)
# ==========================================

import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_video
from PIL import Image
from tqdm.auto import tqdm
import cv2
from transformers import (
    VideoMAEForVideoClassification,
    AutoImageProcessor
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)

import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# CONFIG
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = "/kaggle/input/crime-data/theft"
MODEL_PATH = "/kaggle/input/best-seg-model-v2/pytorch/default/1/best_seg_model_v2.pt"   # your best model
OUT_DIR = "/kaggle/working/final_results"
os.makedirs(OUT_DIR, exist_ok=True)

CLIP_LEN = 16
IMG_SIZE = 224
BATCH_SIZE = 4

CLASSES = ["Burglary", "Robbery", "Shoplifting", "Stealing"]
NUM_CLASSES = len(CLASSES)

# -------------------------
# Dataset
# -------------------------
class VideoDataset(Dataset):
    def __init__(self, root, processor):
        self.samples = []
        self.processor = processor

        for idx, cls in enumerate(CLASSES):
            vids = glob.glob(os.path.join(root, cls, "*.mp4"))
            for v in vids:
                self.samples.append((v, idx))

    def __len__(self):
        return len(self.samples)

    def _read_clip(self, path):
        cap = cv2.VideoCapture(path)
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return [Image.new("RGB", (IMG_SIZE, IMG_SIZE))] * CLIP_LEN

        frames = frames[:CLIP_LEN]
        if len(frames) < CLIP_LEN:
            frames += [frames[-1]] * (CLIP_LEN - len(frames))

        return [Image.fromarray(f) for f in frames]

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frames = self._read_clip(path)
        inputs = self.processor(frames, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), label

# -------------------------
# Load Model
# -------------------------
processor = AutoImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"), strict=False)
model.to(DEVICE)
model.eval()

# -------------------------
# DataLoader
# -------------------------
dataset = VideoDataset(DATA_ROOT, processor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# -------------------------
# Inference
# -------------------------
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for X, y in tqdm(loader, desc="Evaluating"):
        X = X.to(DEVICE)
        outputs = model(X)
        probs = torch.softmax(outputs.logits, dim=1)

        all_labels.extend(y.numpy())
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)
y_prob = np.array(all_probs)

# -------------------------
# Metrics
# -------------------------
acc = accuracy_score(y_true, y_pred)

prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
    y_true, y_pred, average="macro"
)

prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted"
)

print("\n=== FINAL RESULTS (VideoMAE) ===")
print("Accuracy:", acc)
print("Macro F1:", f1_macro)
print("Weighted F1:", f1_weighted)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

# -------------------------
# Confusion Matrix
# -------------------------
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - VideoMAE")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"))
plt.close()

# -------------------------
# ROC & AUC
# -------------------------
plt.figure(figsize=(7,6))
auc_scores = {}

for i, cls in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_prob[:, i])
    auc_score = auc(fpr, tpr)
    auc_scores[cls] = auc_score
    plt.plot(fpr, tpr, label=f"{cls} (AUC={auc_score:.2f})")

plt.plot([0,1],[0,1],"k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - VideoMAE")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png"))
plt.close()

print("\nAUC Scores:")
for k,v in auc_scores.items():
    print(f"{k}: {v:.3f}")

print("\nAll results saved to:", OUT_DIR)

In [ ]:
!pip -q install rouge-score nltk --quiet

# ***Summarization Evaluation of Results***

In [ ]:
# ==========================================================
# STAGE 3 — SUMMARIZATION EVALUATION
# Ground Truth: UCFCrime JSON sentences
# Prediction   : Stage-3 LLM generated paragraphs
# Metrics      : ROUGE-1 / ROUGE-2 / ROUGE-L / BLEU-1 / BLEU-2
# ==========================================================

!pip -q install rouge-score nltk --quiet

import os
import json
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt

# -------------------------
# CONFIG
# -------------------------
UCF_JSON = "/kaggle/input/crime-data/theft/UCFCrime_Val.json"
PRED_DIR = "/kaggle/input/stage3/final_paragraphs_stage32"
OUT_DIR  = "/kaggle/working/stage3_eval_results"
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------
# LOAD UCFCrime GT
# -------------------------
with open(UCF_JSON) as f:
    gt_data = json.load(f)

# -------------------------
# BUILD GT SUMMARIES
# -------------------------
gt_summaries = {}
for vid, meta in gt_data.items():
    sents = meta.get("sentences", [])
    if isinstance(sents, list) and len(sents) > 0:
        gt_summaries[vid] = " ".join(s.strip() for s in sents)

# -------------------------
# LOAD PRED SUMMARIES
# -------------------------
pred_summaries = {}
for f in os.listdir(PRED_DIR):
    if f.endswith("_summary.txt"):
        vid = f.replace("_summary.txt", "")
        with open(os.path.join(PRED_DIR, f)) as fh:
            pred_summaries[vid] = fh.read().strip()

# -------------------------
# ALIGN PAIRS
# -------------------------
gt_texts = []
pred_texts = []

for vid in pred_summaries:
    if vid in gt_summaries:
        gt_texts.append(gt_summaries[vid])
        pred_texts.append(pred_summaries[vid])

print("Paired summaries:", len(gt_texts))

# -------------------------
# ROUGE
# -------------------------
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

r1, r2, rL = [], [], []

for gt, pred in zip(gt_texts, pred_texts):
    scores = scorer.score(gt, pred)
    r1.append(scores["rouge1"].fmeasure)
    r2.append(scores["rouge2"].fmeasure)
    rL.append(scores["rougeL"].fmeasure)

ROUGE_1 = np.mean(r1)
ROUGE_2 = np.mean(r2)
ROUGE_L = np.mean(rL)

# -------------------------
# BLEU
# -------------------------
smooth = SmoothingFunction().method1
b1, b2 = [], []

for gt, pred in zip(gt_texts, pred_texts):
    ref = [gt.split()]
    hyp = pred.split()
    b1.append(sentence_bleu(ref, hyp, weights=(1,0,0,0), smoothing_function=smooth))
    b2.append(sentence_bleu(ref, hyp, weights=(0.5,0.5,0,0), smoothing_function=smooth))

BLEU_1 = np.mean(b1)
BLEU_2 = np.mean(b2)

# -------------------------
# PRINT RESULTS
# -------------------------
print("\n=== STAGE 3 RESULTS — LLM SUMMARIZATION ===")
print(f"ROUGE-1 F1 : {ROUGE_1:.4f}")
print(f"ROUGE-2 F1 : {ROUGE_2:.4f}")
print(f"ROUGE-L F1 : {ROUGE_L:.4f}")
print(f"BLEU-1     : {BLEU_1:.4f}")
print(f"BLEU-2     : {BLEU_2:.4f}")

# -------------------------
# SAVE METRICS
# -------------------------
metrics = {
    "ROUGE-1": ROUGE_1,
    "ROUGE-2": ROUGE_2,
    "ROUGE-L": ROUGE_L,
    "BLEU-1": BLEU_1,
    "BLEU-2": BLEU_2
}

with open(os.path.join(OUT_DIR, "stage3_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# -------------------------
# PLOT ROUGE CURVE
# -------------------------
plt.figure(figsize=(6,4))
plt.bar(["ROUGE-1", "ROUGE-2", "ROUGE-L"], [ROUGE_1, ROUGE_2, ROUGE_L])
plt.ylabel("F1 Score")
plt.title("Stage 3 LLM Summarization Performance")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "stage3_rouge_scores.png"))
plt.close()

print("\nSaved results to:", OUT_DIR)

In [ ]:
!pip install av

# ***VideoMAE Probabilities Result***

In [ ]:
# ==========================================
# Stage 1 — VideoMAE SINGLE VIDEO OUTPUT
# (FOR SCREENSHOT PURPOSE)
# ==========================================

import torch
import cv2
import numpy as np
from PIL import Image
from transformers import VideoMAEForVideoClassification, AutoImageProcessor

# -------------------------
# CONFIG
# -------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VIDEO_PATH = "/kaggle/input/crime-data/theft/Robbery/Robbery007_x264.mp4"
MODEL_PATH = "/kaggle/input/best-seg-model-v2/pytorch/default/1/best_seg_model_v2.pt"

CLASSES = ["Burglary", "Robbery", "Shoplifting", "Stealing"]
CLIP_LEN = 16
IMG_SIZE = 224

# -------------------------
# Load video frames
# -------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []

while len(frames) < CLIP_LEN:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frames.append(Image.fromarray(frame))

cap.release()

if len(frames) < CLIP_LEN:
    frames += [frames[-1]] * (CLIP_LEN - len(frames))

# -------------------------
# Load model
# -------------------------
processor = AutoImageProcessor.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=len(CLASSES),
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"), strict=False)
model.to(DEVICE)
model.eval()

# -------------------------
# Inference
# -------------------------
inputs = processor(frames, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0].cpu().numpy()

pred_idx = probs.argmax()
pred_class = CLASSES[pred_idx]

# -------------------------
# PRINT OUTPUT (THIS IS YOUR SCREENSHOT)
# -------------------------
print("\n===== STAGE 1 OUTPUT (VideoMAE) =====")
print(f"Input Video      : {VIDEO_PATH.split('/')[-1]}")
print(f"Predicted Class  : {pred_class}\n")

print("Class Probabilities:")
for cls, p in zip(CLASSES, probs):
    print(f"  {cls:<12}: {p:.3f}")

print("====================================")